## v4a1

In [1]:
# %% [markdown]
# IDMOD Microsimulation – GPU-Accelerated with cuDF & CuPy (No NumPy/Pandas)
# -----------------------------------------------------------------------
# Full workflow: Modules 1–17 with policy scenarios, deterministic & probabilistic analyses,
# CE-plane, CEAC, EVPI, tornado, subgroup, budget impact, equity,
# PSA histograms, CEAF, input summary, cost-mix stacked area, publication comparison.

# %% [markdown]
## 1. Imports & Setup

import cupy as cp
import cudf
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
from scipy.stats import beta
from concurrent.futures import ThreadPoolExecutor  # <--- New Import

warnings.filterwarnings("ignore")
cp.random.seed(42)

# %% [markdown]
## 2. Global Parameters & GPU Constants

START_YEAR, END_YEAR = 2020, 2100
YEARS = list(range(START_YEAR, END_YEAR + 1))
DISCOUNT = 0.035
POP_SIZE = 62000  # Base population size

# Costs are based on 2017 values from the paper, adjust to START_YEAR (2020)
COST_REFERENCE_YEAR = 2017
INFLATION_RATE = 0.03
INFLATION_FACTOR = (1 + INFLATION_RATE) ** (START_YEAR - COST_REFERENCE_YEAR)

# Demographics
SEVERITY_DIST = cp.array([0.70, 0.30])
AGE_BREAKS = cp.arange(0, 95, 5)
AGE_DIST = cp.array(
    [
        0.064,
        0.067,
        0.069,
        0.069,
        0.071,
        0.076,
        0.080,
        0.077,
        0.080,
        0.079,
        0.073,
        0.078,
        0.077,
        0.072,
        0.055,
        0.041,
        0.031,
        0.022,
        0.012,
    ]
)
AGE_DIST /= AGE_DIST.sum()

# Mortality
LIFE_EXP_CP = cp.array([74, 54])
ABS_MORT = cp.array(
    [
        0.0002,
        0.0001,
        0.0001,
        0.0002,
        0.0003,
        0.0005,
        0.0007,
        0.0011,
        0.0019,
        0.0035,
        0.0055,
        0.0085,
        0.0130,
        0.0200,
        0.0350,
        0.0600,
        0.0950,
        0.1500,
        0.2500,
    ]
)
MORT_HR = cp.array([2.0, 4.0])

defaul_yield = 0.45

# Fertility bands
FERT_AGE_SPEC = {
    (0, (15, 19)): 0.003,
    (0, (20, 24)): 0.008,
    (0, (25, 29)): 0.010,
    (0, (30, 34)): 0.007,
    (0, (35, 45)): 0.002,
    (1, (15, 19)): 0.001,
    (1, (20, 24)): 0.003,
    (1, (25, 29)): 0.004,
    (1, (30, 34)): 0.003,
    (1, (35, 45)): 0.001,
}
# Build GPU lookup table for fertility rates
fert_data = []
for (s, (lo, hi)), r in FERT_AGE_SPEC.items():
    for age in range(lo, hi + 1):
        fert_data.append({"severity": s, "age": age, "rate": r})
FERT_RATES_GPU = cudf.DataFrame(fert_data)

# Utilities
U_BASE_CP = cp.array([0.78, 0.65])
U_DECAY = 0.0005
U_DIAG = 0.02
U_CARER = 0.05

# Additional parameters for WGS diagnostic pathway impact
WGS_DIAG_COST_ADJUSTMENT = -1000  # Placeholder: e.g., representing annual cost savings from fewer diagnostic tests due to WGS
WGS_DIAG_QALY_ADJUSTMENT = 0.01  # Placeholder: e.g., representing additional annual QALY gain from earlier/precise intervention due to WGS

# Costs - Adjusted for inflation from COST_REFERENCE_YEAR to START_YEAR
COST_INP_CP = cp.array([5000, 15000]) * INFLATION_FACTOR
COST_OUTP_CP = cp.array([1000, 3000]) * INFLATION_FACTOR
COST_PHARM_CP = cp.array([200, 500]) * INFLATION_FACTOR
EDU0 = cp.array([16107, 27903, 46733]) * INFLATION_FACTOR
EDU1 = cp.array([18958, 30789, 49604]) * INFLATION_FACTOR
RESPITE = 13024 * INFLATION_FACTOR
MOBILITY = 2493 * INFLATION_FACTOR
WELFARE_CP = cp.array([3000, 8000]) * INFLATION_FACTOR
EMP_CP = cp.array([0, 10000, 0]) * INFLATION_FACTOR

# Morbidity
MORB_KEYS = ["epilepsy", "asd", "mental"]
MORB_PROB = {"epilepsy": 0.25, "asd": 0.30, "mental": 0.35}
MORB_COST_CP = cp.array([2000, 1500, 1000]) * INFLATION_FACTOR
MORB_QALY_CP = cp.array([0.04, 0.03, 0.05])

# %% [markdown]
## 3. Cohort Generator (GPU)


def sample_ages_gpu(n):
    idx = cp.random.choice(len(AGE_BREAKS), n, p=AGE_DIST)
    return (AGE_BREAKS[idx] + cp.random.randint(0, 5, n)).astype(int)


def gen_population_gpu(n, familial_prop, arm="prospective"):
    ages = sample_ages_gpu(n)
    sex = cp.random.choice(2, n)
    sev = cp.random.choice(2, n, p=SEVERITY_DIST)
    familial = cp.random.random(n) < familial_prop
    diag = (
        cp.zeros(n, bool)
        if arm == "prospective"
        else (cp.random.random(n) < defaul_yield)
    )
    living = cp.random.choice(4, n, p=cp.array([0.2, 0.1, 0.3, 0.4]))
    employment = cp.random.choice(3, n, p=cp.array([0.7, 0.2, 0.1]))
    uses_respite = cp.random.random(n) < 0.3
    needs_mobility = cp.random.random(n) < 0.2
    school = cp.where(ages < 5, -1, cp.where(ages < 12, 0, 1))
    support = cp.random.choice(3, n, p=cp.array([0.5, 0.3, 0.2]))
    ses = cp.random.choice(5, n)
    df = cudf.DataFrame(
        {
            "age": ages,
            "sex": sex,
            "severity": sev,
            "familial": familial,
            "diagnosed": diag,
            "living": living,
            "employment": employment,
            "uses_respite": uses_respite,
            "needs_mobility": needs_mobility,
            "school": school,
            "support": support,
            "ses": ses,
        }
    )
    # Add columns for outputs
    for col in [
        "annual_cost",
        "annual_qaly",
        "cost_health",
        "cost_edu",
        "cost_respite",
        "cost_mobility",
        "cost_welfare",
        "cost_emp",
        "cost_morbidity",
    ]:
        df[col] = 0.0
    return df

# %% [markdown]
## 4. Population System (GPU)


class PopulationSystemGPU:
    def __init__(self, pop_size, familial_prop, arm="prospective"):
        self.arm = arm
        self.pop = gen_population_gpu(pop_size, familial_prop, arm)

    def _mortality(self):
        df = self.pop
        ages = df["age"].to_cupy()
        sev = df["severity"].to_cupy()
        # Ensure age index is within bounds for ABS_MORT
        age_indices = cp.clip((ages // 5).astype(int), 0, len(ABS_MORT) - 1)
        surv = ABS_MORT[age_indices] * MORT_HR[sev]
        mask = (cp.random.random(len(df)) > surv) & (ages < LIFE_EXP_CP[sev])
        self.pop = df.iloc[cp.where(mask)[0]].reset_index(drop=True)

    def _fertility(self, familial_prop):
        df = self.pop
        moms = df[(df["sex"] == 0) & df["age"].between(15, 45)]
        if moms.empty:
            return
        mr = moms.merge(FERT_RATES_GPU, on=["severity", "age"], how="left").fillna(0)
        rates = mr["rate"].values
        births = cp.random.random(len(rates)) < rates
        idx = cp.nonzero(births)[0]
        if len(idx) > 0:
            babies = gen_population_gpu(len(idx), familial_prop, self.arm)
            babies["age"] = 0
            # Inherit 'familial' status from the mother
            flags = mr["familial"].iloc[idx.get()].reset_index(drop=True)
            babies["familial"] = flags
            self.pop = cudf.concat([df, babies], ignore_index=True)

    def _diagnosis(self, params):
        if self.arm != "prospective":
            return
        df = self.pop
        mask = (
            (~df["diagnosed"])
            & df["familial"]
            & (df["age"] >= params.get("min_test_age", 0))
        )
        cnt = int(mask.sum())
        if cnt > 0:
            # Use .loc with a boolean mask to assign values
            indices = df.index[mask]
            df.loc[indices, "diagnosed"] = cp.random.random(cnt) < params["yield_wgs"]
        self.pop = df

    def _cost_qaly(self, year, params):
        df = self.pop
        n = len(df)
        if n == 0:
            return

        disc = 1 / ((1 + DISCOUNT) ** (year - START_YEAR))
        sev = df["severity"].to_cupy()
        ages = df["age"].to_cupy()

        # Costs
        hc = (COST_INP_CP[sev] + COST_OUTP_CP[sev] + COST_PHARM_CP[sev]) * disc
        rc = df["uses_respite"].to_cupy().astype(int) * RESPITE * disc
        mc = (
            -df["needs_mobility"].to_cupy().astype(int) * MOBILITY * disc
        )  # Cost saving
        wc = WELFARE_CP[sev] * disc
        emp = EMP_CP[df["employment"].to_cupy()] * disc

        ec = cp.zeros(n, dtype=cp.float64)
        m0 = df["school"] == 0
        m1 = df["school"] == 1
        if not df.loc[m0].empty:
            ec[m0.to_cupy()] = EDU0[df.loc[m0, "support"].to_cupy()] * disc
        if not df.loc[m1].empty:
            ec[m1.to_cupy()] = EDU1[df.loc[m1, "support"].to_cupy()] * disc

        morb_cost = cp.zeros(n, dtype=cp.float64)
        base_rand = cp.random.random(n)
        for i, k in enumerate(MORB_KEYS):
            m = base_rand < MORB_PROB[k]
            morb_cost += m.astype(cp.float64) * MORB_COST_CP[i] * disc

        # Apply WGS-specific cost adjustment if in prospective arm and diagnosed
        if self.arm == "prospective":
            m = df["diagnosed"].to_cupy()
            morb_cost += m.astype(cp.float64) * WGS_DIAG_COST_ADJUSTMENT * disc

        # QALYs
        morb_qaly = cp.zeros(n, dtype=cp.float64)
        for i, k in enumerate(MORB_KEYS):
            m = base_rand < MORB_PROB[k]  # Use same random numbers for consistency
            morb_qaly += m.astype(cp.float64) * MORB_QALY_CP[i] * disc

        uq = U_BASE_CP[sev] * cp.exp(-U_DECAY * cp.clip(ages - 20, 0, None))
        total_qaly = (
            uq + df["diagnosed"].to_cupy().astype(int) * U_DIAG + U_CARER
        ) * disc - morb_qaly

        # Apply WGS-specific QALY adjustment if in prospective arm and diagnosed
        if self.arm == "prospective":
            m = df["diagnosed"].to_cupy()
            total_qaly += m.astype(cp.float64) * WGS_DIAG_QALY_ADJUSTMENT * disc

        # Assign back to DataFrame
        df["cost_health"] = hc
        df["cost_edu"] = ec
        df["cost_respite"] = rc
        df["cost_mobility"] = mc
        df["cost_welfare"] = wc
        df["cost_emp"] = emp
        df["cost_morbidity"] = morb_cost
        df["annual_cost"] = hc + ec + rc + mc + wc + emp + morb_cost
        df["annual_qaly"] = total_qaly
        self.pop = df

    def step(self, year, params):
        self._mortality()
        self._fertility(params["familial_prop"])
        self._diagnosis(params)
        self.pop["age"] += 1
        self._cost_qaly(year, params)

# %% [markdown]
## 5. Run Helpers


def run_sim(
    params_and_idx, pop_size=POP_SIZE
):  # <--- Modified to accept (params, index)
    """Fast runner for aggregate results (dCost, dQALY). Used for PSA, Tornado."""
    params, idx = params_and_idx  # Unpack the tuple
    # For PSA, it's beneficial to set a unique CuPy random seed for each thread/process
    # This avoids potential issues if multiple threads try to use the same global CuPy random state
    # and ensures independent random number sequences for each PSA run.
    cp.random.seed(42 + idx)  # Use the index to make the seed unique

    sim_r = PopulationSystemGPU(pop_size, params["familial_prop"], "retrospective")
    sim_p = PopulationSystemGPU(pop_size, params["familial_prop"], "prospective")

    cum_r_c = cum_r_q = cum_p_c = cum_p_q = 0.0

    for yr in YEARS:
        sim_r.step(yr, params)
        if not sim_r.pop.empty:
            cum_r_c += float(sim_r.pop["annual_cost"].sum())
            cum_r_q += float(sim_r.pop["annual_qaly"].sum())

        sim_p.step(yr, params)
        if not sim_p.pop.empty:
            cum_p_c += float(sim_p.pop["annual_cost"].sum())
            cum_p_q += float(sim_p.pop["annual_qaly"].sum())

    return (cum_p_c - cum_r_c, cum_p_q - cum_r_q)


def run_scenario(params, pop_size=POP_SIZE):
    """Detailed runner that returns aggregate results AND a per-year DataFrame."""
    outputs = []
    final_totals = {}

    for arm in ["retrospective", "prospective"]:
        sim = PopulationSystemGPU(pop_size, params["familial_prop"], arm)
        cum_c = 0.0
        cum_q = 0.0
        for yr in YEARS:
            sim.step(yr, params)
            df = sim.pop

            if df.empty:
                # Append a zero-row to maintain structure
                row_data = {
                    "arm": arm,
                    "year": yr,
                    "annual_cost": 0.0,
                    "cost_health": 0.0,
                    "cost_edu": 0.0,
                    "cost_respite": 0.0,
                    "cost_mobility": 0.0,
                    "cost_welfare": 0.0,
                    "cost_emp": 0.0,
                    "cost_morbidity": 0.0,
                    "costs": cum_c,
                    "qalys": cum_q,
                    **{f"ses_q{i}_pct": 0.0 for i in range(5)},
                }
                outputs.append(row_data)
                continue

            # Extract already-discounted annual components for detailed output
            hc = float(df["cost_health"].sum())
            edu = float(df["cost_edu"].sum())
            rc = float(df["cost_respite"].sum())
            mob = float(df["cost_mobility"].sum())
            wc = float(df["cost_welfare"].sum())
            emp = float(df["cost_emp"].sum())
            mc = float(df["cost_morbidity"].sum())

            annual_c_sum = float(df["annual_cost"].sum())
            annual_q_sum = float(df["annual_qaly"].sum())

            cum_c += annual_c_sum
            cum_q += annual_q_sum

            outputs.append(
                {
                    "arm": arm,
                    "year": yr,
                    "annual_cost": annual_c_sum,  # Crucial for budget impact
                    "cost_health": hc,
                    "cost_edu": edu,
                    "cost_respite": rc,
                    "cost_mobility": mob,
                    "cost_welfare": wc,
                    "cost_emp": emp,
                    "cost_morbidity": mc,
                    "costs": cum_c,
                    "qalys": cum_q,
                    **{
                        f"ses_q{i}_pct": float((df["ses"] == i).mean() * 100)
                        for i in range(5)
                    },
                }
            )

    final_totals[arm] = {"cost": cum_c, "qaly": cum_q}

    d_cost = final_totals["prospective"]["cost"] - final_totals["retrospective"]["cost"]
    d_qaly = final_totals["prospective"]["qaly"] - final_totals["retrospective"]["qaly"]

    return (d_cost, d_qaly), cudf.DataFrame(outputs)

# %% [markdown]
## 6. Deterministic Scenarios

print("--- Running Module 6: Deterministic Scenarios ---")
scenarios = {"low": 0.10, "base": 0.165, "high": 0.23}
results = []
for name, val in scenarios.items():
    # Pass a dummy index (0) since run_sim expects it now, but it's not used in this context
    # unless you want to vary random seeds for deterministic runs too.
    # For deterministic, it's common to use a fixed seed or no explicit seed change.
    dc, dq = run_sim(
        ({"familial_prop": val, "yield_wgs": defaul_yield}, 0), pop_size=20000
    )
    icer = (dc / dq) if dq != 0 else 0
    results.append({"scenario": name, "dCost": dc, "dQALY": dq, "ICER": icer})

det_df = cudf.DataFrame(results)
det_df.to_csv("deterministic_summary.csv", index=False)
print(det_df)

plt.figure()
plt.bar(
    det_df["scenario"].to_arrow().to_pylist(), det_df["ICER"].to_arrow().to_pylist()
)
# Corrected Line: Removed the '$' characters which can cause copy-paste errors.
plt.title("Deterministic ICERs (/QALY)")
plt.ylabel("ICER (/QALY)")
plt.show()

# %% [markdown]
## 7. Probabilistic Sensitivity Analysis (PSA)

print("\n--- Running Module 7: Probabilistic Sensitivity Analysis ---")
psa_n = 300
psa_pop_size = 8000  # Use a specific population size for PSA
# Generate all parameters upfront
psa_params_list = []
for i in range(psa_n):
    samp = {"familial_prop": beta.rvs(30, 150), "yield_wgs": beta.rvs(45, 55)}
    psa_params_list.append((samp, i))  # Store parameters and index


# Define a helper function to pass parameters and pop_size to run_sim
def run_sim_with_fixed_pop_size(params_and_idx):
    return run_sim(params_and_idx, pop_size=psa_pop_size)


# Use ThreadPoolExecutor for parallel execution
# max_workers can be adjusted based on your CPU cores and GPU capacity
# Typically, max_workers = number of CPU cores is a good starting point.
# If your GPU is powerful enough to handle multiple simultaneous small pop_size simulations,
# you might increase this, but be mindful of GPU memory and contention.
num_workers = 4  # Example: adjust based on your system (e.g., os.cpu_count())

psa_out = []
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    # `executor.map` applies run_sim_with_fixed_pop_size to each item in psa_params_list
    # and returns results in the order the tasks were submitted.
    for i, result in enumerate(
        executor.map(run_sim_with_fixed_pop_size, psa_params_list)
    ):
        dc, dq = result
        psa_out.append({"run": i, "dCost": dc, "dQALY": dq})
        if (i + 1) % 50 == 0:
            print(f"PSA Run {i + 1}/{psa_n} complete.")

psa_df = cudf.DataFrame(psa_out)
psa_df.to_csv("psa_samples.csv", index=False)
print("PSA results saved to psa_samples.csv")

# %% [markdown]
## 8. CE-Plane, CEAC & EVPI

print("\n--- Running Module 8: CE-Plane, CEAC & EVPI ---")
wtp = cp.linspace(0, 150000, 40)
dcost = psa_df["dCost"].to_cupy()
dqaly = psa_df["dQALY"].to_cupy()

# Filter out non-positive QALYs for stable NB calculation
valid_psa = dqaly > 0
nb = dqaly[valid_psa, None] * wtp[None, :] - dcost[valid_psa, None]

# CE-Plane scatter
plt.figure()
plt.scatter(dqaly.get(), dcost.get(), alpha=0.6)
plt.xlabel("Incremental QALYs (Delta QALY)")
plt.ylabel("Incremental Cost (Delta Cost)")
plt.title("Cost-Effectiveness Plane")
plt.grid(True)
plt.show()

# CEAC
prob = (nb > 0).mean(axis=0)
plt.figure()
plt.plot(wtp.get(), prob.get())
plt.title("Cost-Effectiveness Acceptability Curve (CEAC)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Probability Cost-Effective")
plt.ylim(0, 1)
plt.grid(True)
plt.show()

# EVPI - Corrected formula
evpi = cp.mean(cp.maximum(0, nb), axis=0) - cp.maximum(0, cp.mean(nb, axis=0))
plt.figure()
plt.plot(wtp.get(), evpi.get())
plt.title("Expected Value of Perfect Information (EVPI)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("EVPI")
plt.grid(True)
plt.show()

# %% [markdown]
## 9. One-Way Sensitivity (Tornado)

print("\n--- Running Module 9: One-Way Sensitivity (Tornado) ---")
tornado = []
base_params = {"familial_prop": 0.165, "yield_wgs": defaul_yield}
base_dc, base_dq = run_sim((base_params, 0), pop_size=10000)  # Pass dummy index
base_icer = base_dc / base_dq if base_dq != 0 else 0

for param in ["yield_wgs", "familial_prop"]:
    for change in [-0.2, 0.2]:
        p = base_params.copy()
        p[param] = p[param] * (1 + change)
        dc, dq = run_sim((p, 0), pop_size=10000)  # Pass dummy index
        icer = (dc / dq) if dq != 0 else 0
        tornado.append(
            {"param": param, "change": f"{int(change * 100)}%", "ICER": icer}
        )

tor_df = cudf.DataFrame(tornado)
labels = [
    f"{p} {c}"
    for p, c in zip(
        tor_df["param"].to_arrow().to_pylist(), tor_df["change"].to_arrow().to_pylist()
    )
]
icers = tor_df["ICER"].to_arrow().to_pylist()
bar_values = [(x - base_icer) for x in icers]
colors = ["red" if x < 0 else "green" for x in bar_values]

plt.figure()
plt.barh(labels, bar_values, color=colors)
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("Change in ICER from base (/QALY)")
plt.title("Tornado Sensitivity Analysis")
plt.tight_layout()
plt.show()

# %% [markdown]
## 10. Sub-group Analyses

print("\n--- Running Module 10: Sub-group Analyses ---")
sg = []

# 1. Severity
orig_sev_dist = SEVERITY_DIST.copy()
for sev, label in [(0, "Mild"), (1, "Moderate")]:
    SEVERITY_DIST = cp.array([1.0 if i == sev else 0.0 for i in range(2)])
    dc, dq = run_sim(
        ({"familial_prop": 0.165, "yield_wgs": defaul_yield}, 0), pop_size=15000
    )  # Pass dummy index
    sg.append({"group": f"Severity: {label}", "ICER": (dc / dq) if dq != 0 else 0})
SEVERITY_DIST = orig_sev_dist  # Restore

# 2. Age at test
for age in [0, 18, 30]:
    params = {"familial_prop": 0.165, "yield_wgs": defaul_yield, "min_test_age": age}
    dc, dq = run_sim((params, 0), pop_size=15000)  # Pass dummy index
    sg.append({"group": f"Min Test Age: {age}", "ICER": (dc / dq) if dq != 0 else 0})

# 3. Time-horizon
# Note: run_scenario still needs to be run sequentially as it collects detailed yearly data
(dc_full, dq_full), int_full = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=15000
)
dfp = int_full[int_full["arm"] == "prospective"]
dfr = int_full[int_full["arm"] == "retrospective"]
for yrs_n in [10, 20, len(YEARS)]:
    yr = START_YEAR + yrs_n - 1
    c1 = float(dfp.loc[dfp["year"] == yr, "costs"].values[0])
    c0 = float(dfr.loc[dfr["year"] == yr, "costs"].values[0])
    q1 = float(dfp.loc[dfp["year"] == yr, "qalys"].values[0])
    q0 = float(dfr.loc[dfr["year"] == yr, "qalys"].values[0])
    dc, dq = (c1 - c0), (q1 - q0)
    sg.append({"group": f"Horizon: {yrs_n} yrs", "ICER": (dc / dq) if dq != 0 else 0})

# Plot
sg_df = cudf.DataFrame(sg)
plt.figure()
plt.scatter(
    sg_df["ICER"].to_arrow().to_pylist(),
    sg_df["group"].to_arrow().to_pylist(),
    marker="o",
)
plt.axvline(
    base_icer, color="grey", linestyle="--", label=f"Base Case ICER ({base_icer:,.0f})"
)
plt.xlabel("ICER (/QALY)")
plt.title("Sub-group Analyses")
plt.legend()
plt.tight_layout()
plt.grid(axis="x")
plt.show()

# %% [markdown]
## 11. Cost-Mix Stacked Area (Prospective)

print("\n--- Running Module 11: Cost-Mix Stacked Area ---")
(_, _), mix = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
pm = mix[mix["arm"] == "prospective"]
cost_cols = [
    "cost_health",
    "cost_edu",
    "cost_respite",
    "cost_mobility",
    "cost_welfare",
    "cost_emp",
    "cost_morbidity",
]
years_host = pm["year"].to_arrow().to_pylist()
# Un-cumulate the data for stacking annual costs
data = [pm[col].to_arrow().to_pylist() for col in cost_cols]

plt.figure(figsize=(10, 6))
plt.stackplot(
    years_host, *data, labels=[c.replace("cost_", "").capitalize() for c in cost_cols]
)
plt.legend(loc="upper left")
plt.title("Cost Mix Over Time (Prospective Arm - Discounted Annual Costs)")
plt.xlabel("Year")
plt.ylabel("Annual Cost")
plt.show()

# %% [markdown]
## 12. 5-Year Budget Impact

print("\n--- Running Module 12: 5-Year Budget Impact ---")
budg = []
uptakes = [0.2, 0.5, 0.8]
yrs = YEARS[:5]
# Run simulation once to get the cost data
(_, _), df = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
dfp = df[df["arm"] == "prospective"]
dfr = df[df["arm"] == "retrospective"]

for u in uptakes:
    for yr in yrs:
        # Use 'annual_cost' which is not cumulative
        # Correct way to extract a single scalar from a cuDF Series
        c1 = float(dfp.loc[dfp["year"] == yr, "annual_cost"].to_cupy().item())
        c0 = float(dfr.loc[dfr["year"] == yr, "annual_cost"].to_cupy().item())

        # Budget impact is the incremental cost for the new intervention, scaled by uptake
        incremental_cost = c1 - c0
        budg.append({"uptake": u, "year": yr, "budget_impact": incremental_cost * u})

budg_df = cudf.DataFrame(budg)
plt.figure()

# %% [markdown]
## 13. Equity by SES Quintile

print("\n--- Running Module 13: Equity by SES Quintile ---")
(_, _), eq_int = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
bas = eq_int[eq_int["arm"] == "prospective"]
# Select the last year of the simulation
row_series = bas[bas["year"] == YEARS[-1]].iloc[0]

ses_cols = [f"ses_q{i}_pct" for i in range(5)]
# Extract scalar values from the series by converting to cupy and then using .item()
ses_vals = [float(row_series[col].to_cupy().item()) for col in ses_cols]
labels = [f"Quintile {i + 1}" for i in range(5)]

plt.figure()
plt.bar(labels, ses_vals)
plt.title(f"SES Distribution of Population in Final Year ({YEARS[-1]})")
plt.xlabel("SES Quintile")
plt.ylabel("Percentage of Population (%)")
plt.tight_layout()
plt.show()

# %% [markdown]
## 14. PSA Histograms

print("\n--- Running Module 14: PSA Histograms ---")
cost_list = psa_df["dCost"].to_arrow().to_pylist()
qaly_list = psa_df["dQALY"].to_arrow().to_pylist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.hist(cost_list, bins=30, color="skyblue", edgecolor="black")
ax1.set_title("PSA Distribution of Incremental Cost")
ax1.set_xlabel("Delta Cost")
ax1.set_ylabel("Frequency")

ax2.hist(qaly_list, bins=30, color="salmon", edgecolor="black")
ax2.set_title("PSA Distribution of Incremental QALYs")
ax2.set_xlabel("Delta QALY")
ax2.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# %% [markdown]
## 15. CEAF (Cost-Effectiveness Acceptability Frontier)

print("\n--- Running Module 15: CEAF ---")
# Reuse 'prob' (CEAC probabilities) from Module 8
prob_host = prob.get()  # Get the CuPy array to host memory (NumPy array)
ceaf = np.maximum.accumulate(prob_host)  # Use numpy for the accumulate operation

plt.figure()
plt.plot(wtp.get(), prob.get(), label="CEAC")
plt.plot(wtp.get(), ceaf, label="CEAF", linestyle="--")  # ceaf is already a NumPy array
plt.title("CEAC and CEAF")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Probability")
plt.legend()
plt.grid(True)
plt.show()

# %% [markdown]
## 16. Input Parameter Summary

print("\n--- Running Module 16: Saving Input Parameter Summary ---")
params_summary = {
    "DISCOUNT": DISCOUNT,
    "default_yield": defaul_yield,
    "U_DIAG": U_DIAG,
    "U_CARER": U_CARER,
    "RESPITE": RESPITE,
    "MOBILITY": MOBILITY,
    "FERT_AGE_SPEC": json.dumps(
        {f"{k[0]}_{k[1][0]}_{k[1][1]}": v for k, v in FERT_AGE_SPEC.items()}
    ),
    "MORB_PROB": json.dumps(MORB_PROB),
    "AGE_DIST": json.dumps(AGE_DIST.get().tolist()),
    "COST_REFERENCE_YEAR": COST_REFERENCE_YEAR,
    "INFLATION_RATE": INFLATION_RATE,
    "INFLATION_FACTOR": float(INFLATION_FACTOR),  # Corrected line: removed .get()
}

param_df = cudf.DataFrame(
    {
        "parameter": list(params_summary.keys()),
        "value": [str(v) for v in params_summary.values()],
    }
)

param_df.to_csv("input_summary.csv", index=False)
print("Input parameters saved to input_summary.csv")

# %% [markdown]
## 17. Publication Comparison

print("\n--- Running Module 17: Publication Comparison ---")
# Published point estimates for comparison
published = {"base": {"dCost": 480000, "dQALY": 6.5, "ICER": 73846}}

pub_rows = []
for scen, vals in published.items():
    pub_rows.append(
        {
            "scenario": scen,
            "dCost_pub": vals["dCost"],
            "dQALY_pub": vals["dQALY"],
            "ICER_pub": vals["ICER"],
        }
    )
pub_df = cudf.DataFrame(pub_rows)

# Merge with your deterministic simulation output (det_df from Module 6)
df_cmp = det_df.merge(pub_df, on="scenario", how="left")
df_cmp = df_cmp.rename(
    columns={"dCost": "dCost_sim", "dQALY": "dQALY_sim", "ICER": "ICER_sim"}
)

print("Comparison of Simulation vs. Published Data:")
print(df_cmp.to_pandas().to_string())  # Use pandas for cleaner console printing
df_cmp.to_csv("publication_comparison.csv", index=False)
print("Saved comparison to publication_comparison.csv")

# %% [markdown]
## 18. Familial ID Prevalence Sensitivity Analysis

print("\n--- Running Module 18: Familial ID Prevalence Sensitivity Analysis ---")
familial_prevalence_range = cp.linspace(0.10, 0.23, 10).get()  # Range from 10% to 23%
familial_sens_results = []
for f_prop in familial_prevalence_range:
    dc, dq = run_sim(
        {"familial_prop": f_prop, "yield_wgs": defaul_yield}, pop_size=10000
    )
    icer = (dc / dq) if dq != 0 else 0
    familial_sens_results.append(
        {"familial_prop": f_prop, "dCost": dc, "dQALY": dq, "ICER": icer}
    )

familial_sens_df = cudf.DataFrame(familial_sens_results)
print("Familial ID Prevalence Sensitivity Results:")
print(familial_sens_df.to_pandas().to_string())

plt.figure(figsize=(8, 6))
plt.plot(
    familial_sens_df["familial_prop"].to_arrow().to_pylist(),
    familial_sens_df["ICER"].to_arrow().to_pylist(),
    marker="o",
    linestyle="-",
)
plt.xlabel("Familial ID Prevalence")
plt.ylabel("ICER (/QALY)")
plt.title("Sensitivity to Familial ID Prevalence")
plt.grid(True)
plt.show()

## v4a2

In [1]:
# %% [markdown]
# IDMOD Microsimulation – GPU-Accelerated with cuDF & CuPy (No NumPy/Pandas)
# -----------------------------------------------------------------------
# Full workflow: Modules 1–17 with policy scenarios, deterministic & probabilistic analyses,
# CE-plane, CEAC, EVPI, tornado, subgroup, budget impact, equity,
# PSA histograms, CEAF, input summary, cost-mix stacked area, publication comparison.

# %% [markdown]
## 1. Imports & Setup

import cupy as cp
import cudf
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
from concurrent.futures import ThreadPoolExecutor

# New JAX imports
import jax
import jax.numpy as jnp
import jax.random

warnings.filterwarnings("ignore")
cp.random.seed(42)
# Initialize a JAX random key
jax_key = jax.random.PRNGKey(42)  # <--- JAX random key for parameter sampling

# %% [markdown]
## 2. Global Parameters & GPU Constants

START_YEAR, END_YEAR = 2020, 2100
YEARS = list(range(START_YEAR, END_YEAR + 1))
DISCOUNT = 0.035
POP_SIZE = 62000  # Base population size

# Costs are based on 2017 values from the paper, adjust to START_YEAR (2020)
COST_REFERENCE_YEAR = 2017
INFLATION_RATE = 0.03
INFLATION_FACTOR = (1 + INFLATION_RATE) ** (START_YEAR - COST_REFERENCE_YEAR)

# Demographics
SEVERITY_DIST = cp.array([0.70, 0.30])
AGE_BREAKS = cp.arange(0, 95, 5)
AGE_DIST = cp.array(
    [
        0.064,
        0.067,
        0.069,
        0.069,
        0.071,
        0.076,
        0.080,
        0.077,
        0.080,
        0.079,
        0.073,
        0.078,
        0.077,
        0.072,
        0.055,
        0.041,
        0.031,
        0.022,
        0.012,
    ]
)
AGE_DIST /= AGE_DIST.sum()

# Mortality
LIFE_EXP_CP = cp.array([74, 54])
ABS_MORT = cp.array(
    [
        0.0002,
        0.0001,
        0.0001,
        0.0002,
        0.0003,
        0.0005,
        0.0007,
        0.0011,
        0.0019,
        0.0035,
        0.0055,
        0.0085,
        0.0130,
        0.0200,
        0.0350,
        0.0600,
        0.0950,
        0.1500,
        0.2500,
    ]
)
MORT_HR = cp.array([2.0, 4.0])

defaul_yield = 0.45

# Fertility bands
FERT_AGE_SPEC = {
    (0, (15, 19)): 0.003,
    (0, (20, 24)): 0.008,
    (0, (25, 29)): 0.010,
    (0, (30, 34)): 0.007,
    (0, (35, 45)): 0.002,
    (1, (15, 19)): 0.001,
    (1, (20, 24)): 0.003,
    (1, (25, 29)): 0.004,
    (1, (30, 34)): 0.003,
    (1, (35, 45)): 0.001,
}
# Build GPU lookup table for fertility rates
fert_data = []
for (s, (lo, hi)), r in FERT_AGE_SPEC.items():
    for age in range(lo, hi + 1):
        fert_data.append({"severity": s, "age": age, "rate": r})
FERT_RATES_GPU = cudf.DataFrame(fert_data)

# Utilities
U_BASE_CP = cp.array([0.78, 0.65])
U_DECAY = 0.0005
U_DIAG = 0.02
U_CARER = 0.05

# Additional parameters for WGS diagnostic pathway impact
WGS_DIAG_COST_ADJUSTMENT = -1000  # Placeholder: e.g., representing annual cost savings from fewer diagnostic tests due to WGS
WGS_DIAG_QALY_ADJUSTMENT = 0.01  # Placeholder: e.g., representing additional annual QALY gain from earlier/precise intervention due to WGS

# Costs - Adjusted for inflation from COST_REFERENCE_YEAR to START_YEAR
COST_INP_CP = cp.array([5000, 15000]) * INFLATION_FACTOR
COST_OUTP_CP = cp.array([1000, 3000]) * INFLATION_FACTOR
COST_PHARM_CP = cp.array([200, 500]) * INFLATION_FACTOR
EDU0 = cp.array([16107, 27903, 46733]) * INFLATION_FACTOR
EDU1 = cp.array([18958, 30789, 49604]) * INFLATION_FACTOR
RESPITE = 13024 * INFLATION_FACTOR
MOBILITY = 2493 * INFLATION_FACTOR
WELFARE_CP = cp.array([3000, 8000]) * INFLATION_FACTOR
EMP_CP = cp.array([0, 10000, 0]) * INFLATION_FACTOR

# Morbidity
MORB_KEYS = ["epilepsy", "asd", "mental"]
MORB_PROB = {"epilepsy": 0.25, "asd": 0.30, "mental": 0.35}
MORB_COST_CP = cp.array([2000, 1500, 1000]) * INFLATION_FACTOR
MORB_QALY_CP = cp.array([0.04, 0.03, 0.05])

# %% [markdown]
## 3. Cohort Generator (GPU)


def sample_ages_gpu(n):
    idx = cp.random.choice(len(AGE_BREAKS), n, p=AGE_DIST)
    return (AGE_BREAKS[idx] + cp.random.randint(0, 5, n)).astype(int)


def gen_population_gpu(n, familial_prop, arm="prospective"):
    ages = sample_ages_gpu(n)
    sex = cp.random.choice(2, n)
    sev = cp.random.choice(2, n, p=SEVERITY_DIST)
    familial = cp.random.random(n) < familial_prop
    diag = (
        cp.zeros(n, bool)
        if arm == "prospective"
        else (cp.random.random(n) < defaul_yield)
    )
    living = cp.random.choice(4, n, p=cp.array([0.2, 0.1, 0.3, 0.4]))
    employment = cp.random.choice(3, n, p=cp.array([0.7, 0.2, 0.1]))
    uses_respite = cp.random.random(n) < 0.3
    needs_mobility = cp.random.random(n) < 0.2
    school = cp.where(ages < 5, -1, cp.where(ages < 12, 0, 1))
    support = cp.random.choice(3, n, p=cp.array([0.5, 0.3, 0.2]))
    ses = cp.random.choice(5, n)
    df = cudf.DataFrame(
        {
            "age": ages,
            "sex": sex,
            "severity": sev,
            "familial": familial,
            "diagnosed": diag,
            "living": living,
            "employment": employment,
            "uses_respite": uses_respite,
            "needs_mobility": needs_mobility,
            "school": school,
            "support": support,
            "ses": ses,
        }
    )
    # Add columns for outputs
    for col in [
        "annual_cost",
        "annual_qaly",
        "cost_health",
        "cost_edu",
        "cost_respite",
        "cost_mobility",
        "cost_welfare",
        "cost_emp",
        "cost_morbidity",
    ]:
        df[col] = 0.0
    return df

# %% [markdown]
## 4. Population System (GPU)


class PopulationSystemGPU:
    def __init__(self, pop_size, familial_prop, arm="prospective"):
        self.arm = arm
        self.pop = gen_population_gpu(pop_size, familial_prop, arm)

    def _mortality(self):
        df = self.pop
        ages = df["age"].to_cupy()
        sev = df["severity"].to_cupy()
        # Ensure age index is within bounds for ABS_MORT
        age_indices = cp.clip((ages // 5).astype(int), 0, len(ABS_MORT) - 1)
        surv = ABS_MORT[age_indices] * MORT_HR[sev]
        mask = (cp.random.random(len(df)) > surv) & (ages < LIFE_EXP_CP[sev])
        self.pop = df.iloc[cp.where(mask)[0]].reset_index(drop=True)

    def _fertility(self, familial_prop):
        df = self.pop
        moms = df[(df["sex"] == 0) & df["age"].between(15, 45)]
        if moms.empty:
            return
        mr = moms.merge(FERT_RATES_GPU, on=["severity", "age"], how="left").fillna(0)
        rates = mr["rate"].values
        births = cp.random.random(len(rates)) < rates
        idx = cp.nonzero(births)[0]
        if len(idx) > 0:
            babies = gen_population_gpu(len(idx), familial_prop, self.arm)
            babies["age"] = 0
            # Inherit 'familial' status from the mother
            flags = mr["familial"].iloc[idx.get()].reset_index(drop=True)
            babies["familial"] = flags
            self.pop = cudf.concat([df, babies], ignore_index=True)

    def _diagnosis(self, params):
        if self.arm != "prospective":
            return
        df = self.pop
        mask = (
            (~df["diagnosed"])
            & df["familial"]
            & (df["age"] >= params.get("min_test_age", 0))
        )
        cnt = int(mask.sum())
        if cnt > 0:
            # Use .loc with a boolean mask to assign values
            indices = df.index[mask]
            df.loc[indices, "diagnosed"] = cp.random.random(cnt) < params["yield_wgs"]
        self.pop = df

    def _cost_qaly(self, year, params):
        df = self.pop
        n = len(df)
        if n == 0:
            return

        disc = 1 / ((1 + DISCOUNT) ** (year - START_YEAR))
        sev = df["severity"].to_cupy()
        ages = df["age"].to_cupy()

        # Costs
        hc = (COST_INP_CP[sev] + COST_OUTP_CP[sev] + COST_PHARM_CP[sev]) * disc
        rc = df["uses_respite"].to_cupy().astype(int) * RESPITE * disc
        mc = (
            -df["needs_mobility"].to_cupy().astype(int) * MOBILITY * disc
        )  # Cost saving
        wc = WELFARE_CP[sev] * disc
        emp = EMP_CP[df["employment"].to_cupy()] * disc

        ec = cp.zeros(n, dtype=cp.float64)
        m0 = df["school"] == 0
        m1 = df["school"] == 1
        if not df.loc[m0].empty:
            ec[m0.to_cupy()] = EDU0[df.loc[m0, "support"].to_cupy()] * disc
        if not df.loc[m1].empty:
            ec[m1.to_cupy()] = EDU1[df.loc[m1, "support"].to_cupy()] * disc

        morb_cost = cp.zeros(n, dtype=cp.float64)
        base_rand = cp.random.random(n)
        for i, k in enumerate(MORB_KEYS):
            m = base_rand < MORB_PROB[k]
            morb_cost += m.astype(cp.float64) * MORB_COST_CP[i] * disc

        # Apply WGS-specific cost adjustment if in prospective arm and diagnosed
        if self.arm == "prospective":
            m = df["diagnosed"].to_cupy()
            morb_cost += m.astype(cp.float64) * WGS_DIAG_COST_ADJUSTMENT * disc

        # QALYs
        morb_qaly = cp.zeros(n, dtype=cp.float64)
        for i, k in enumerate(MORB_KEYS):
            m = base_rand < MORB_PROB[k]  # Use same random numbers for consistency
            morb_qaly += m.astype(cp.float64) * MORB_QALY_CP[i] * disc

        uq = U_BASE_CP[sev] * cp.exp(-U_DECAY * cp.clip(ages - 20, 0, None))
        total_qaly = (
            uq + df["diagnosed"].to_cupy().astype(int) * U_DIAG + U_CARER
        ) * disc - morb_qaly

        # Apply WGS-specific QALY adjustment if in prospective arm and diagnosed
        if self.arm == "prospective":
            m = df["diagnosed"].to_cupy()
            total_qaly += m.astype(cp.float64) * WGS_DIAG_QALY_ADJUSTMENT * disc

        # Assign back to DataFrame
        df["cost_health"] = hc
        df["cost_edu"] = ec
        df["cost_respite"] = rc
        df["cost_mobility"] = mc
        df["cost_welfare"] = wc
        df["cost_emp"] = emp
        df["cost_morbidity"] = morb_cost
        df["annual_cost"] = hc + ec + rc + mc + wc + emp + morb_cost
        df["annual_qaly"] = total_qaly
        self.pop = df

    def step(self, year, params):
        self._mortality()
        self._fertility(params["familial_prop"])
        self._diagnosis(params)
        self.pop["age"] += 1
        self._cost_qaly(year, params)

# %% [markdown]
## 5. Run Helpers


def run_sim(params, pop_size=POP_SIZE, seed_idx=None):
    """Fast runner for aggregate results (dCost, dQALY). Used for PSA, Tornado.
    Accepts an optional seed_idx for probabilistic runs."""
    if seed_idx is not None:
        cp.random.seed(42 + seed_idx)
    else:
        # Default seed for deterministic/non-PSA runs for reproducibility
        cp.random.seed(42)

    sim_r = PopulationSystemGPU(pop_size, params["familial_prop"], "retrospective")
    sim_p = PopulationSystemGPU(pop_size, params["familial_prop"], "prospective")

    cum_r_c = cum_r_q = cum_p_c = cum_p_q = 0.0

    for yr in YEARS:
        sim_r.step(yr, params)
        if not sim_r.pop.empty:
            cum_r_c += float(sim_r.pop["annual_cost"].sum())
            cum_r_q += float(sim_r.pop["annual_qaly"].sum())

        sim_p.step(yr, params)
        if not sim_p.pop.empty:
            cum_p_c += float(sim_p.pop["annual_cost"].sum())
            cum_p_q += float(sim_p.pop["annual_qaly"].sum())

    return (cum_p_c - cum_r_c, cum_p_q - cum_r_q)


def run_scenario(params, pop_size=POP_SIZE):
    """Detailed runner that returns aggregate results AND a per-year DataFrame."""
    outputs = []
    final_totals = {}

    for arm in ["retrospective", "prospective"]:
        sim = PopulationSystemGPU(pop_size, params["familial_prop"], arm)
        cum_c = 0.0
        cum_q = 0.0
        for yr in YEARS:
            sim.step(yr, params)
            df = sim.pop

            if df.empty:
                # Append a zero-row to maintain structure
                row_data = {
                    "arm": arm,
                    "year": yr,
                    "annual_cost": 0.0,
                    "cost_health": 0.0,
                    "cost_edu": 0.0,
                    "cost_respite": 0.0,
                    "cost_mobility": 0.0,
                    "cost_welfare": 0.0,
                    "cost_emp": 0.0,
                    "cost_morbidity": 0.0,
                    "costs": cum_c,
                    "qalys": cum_q,
                    **{f"ses_q{i}_pct": 0.0 for i in range(5)},
                }
                outputs.append(row_data)
                continue

            # Extract already-discounted annual components for detailed output
            hc = float(df["cost_health"].sum())
            edu = float(df["cost_edu"].sum())
            rc = float(df["cost_respite"].sum())
            mob = float(df["cost_mobility"].sum())
            wc = float(df["cost_welfare"].sum())
            emp = float(df["cost_emp"].sum())
            mc = float(df["cost_morbidity"].sum())

            annual_c_sum = float(df["annual_cost"].sum())
            annual_q_sum = float(df["annual_qaly"].sum())

            cum_c += annual_c_sum
            cum_q += annual_q_sum

            outputs.append(
                {
                    "arm": arm,
                    "year": yr,
                    "annual_cost": annual_c_sum,  # Crucial for budget impact
                    "cost_health": hc,
                    "cost_edu": edu,
                    "cost_respite": rc,
                    "cost_mobility": mob,
                    "cost_welfare": wc,
                    "cost_emp": emp,
                    "cost_morbidity": mc,
                    "costs": cum_c,
                    "qalys": cum_q,
                    **{
                        f"ses_q{i}_pct": float((df["ses"] == i).mean() * 100)
                        for i in range(5)
                    },
                }
            )

        final_totals[arm] = {"cost": cum_c, "qaly": cum_q}

    d_cost = final_totals["prospective"]["cost"] - final_totals["retrospective"]["cost"]
    d_qaly = final_totals["prospective"]["qaly"] - final_totals["retrospective"]["qaly"]

    return (d_cost, d_qaly), cudf.DataFrame(outputs)

# %% [markdown]
## 6. Deterministic Scenarios

print("--- Running Module 6: Deterministic Scenarios ---")
scenarios = {"low": 0.10, "base": 0.165, "high": 0.23}
results = []
for name, val in scenarios.items():
    # Pass a dummy index (0) since run_sim expects it now, but it's not used in this context
    # For deterministic, it's common to use a fixed seed or no explicit seed change.
    dc, dq = run_sim(
        ({"familial_prop": val, "yield_wgs": defaul_yield}, 0), pop_size=20000
    )
    icer = (dc / dq) if dq != 0 else 0
    results.append({"scenario": name, "dCost": dc, "dQALY": dq, "ICER": icer})

det_df = cudf.DataFrame(results)
det_df.to_csv("deterministic_summary.csv", index=False)
print(det_df)

plt.figure()
plt.bar(
    det_df["scenario"].to_arrow().to_pylist(), det_df["ICER"].to_arrow().to_pylist()
)
plt.title("Deterministic ICERs (/QALY)")
plt.ylabel("ICER (/QALY)")
plt.show()

# %% [markdown]
## 7. Probabilistic Sensitivity Analysis (PSA)

print("\n--- Running Module 7: Probabilistic Sensitivity Analysis ---")
psa_n = 300
psa_pop_size = 8000  # Use a specific population size for PSA

# Generate JAX keys for all PSA samples upfront
jax_keys_for_psa = jax.random.split(jax_key, psa_n)


# Use jax.vmap to vectorize the beta sampling over the array of keys
@jax.jit
def sample_params_single_key(key):
    # This function takes a single key and returns a tuple of (familial_prop, yield_wgs)
    # The 'shape' argument is for the shape of the *output* of this single sample, so it's ()
    familial_prop = jax.random.beta(key, a=30, b=150, shape=())
    key, subkey = jax.random.split(
        key
    )  # Split key for the next random call within this function
    yield_wgs = jax.random.beta(subkey, a=45, b=55, shape=())
    return familial_prop, yield_wgs


# Apply vmap to sample_params_single_key over the first axis of jax_keys_for_psa
# The result will be two JAX arrays, each of shape (psa_n,)
familial_props_jax, yield_wgs_jax = jax.vmap(sample_params_single_key)(jax_keys_for_psa)

# Transfer parameters to CPU as NumPy arrays for easier iteration and passing to threads
familial_props_np = np.array(familial_props_jax)
yield_wgs_np = np.array(yield_wgs_jax)

psa_params_list = []
for i in range(psa_n):
    samp = {"familial_prop": familial_props_np[i], "yield_wgs": yield_wgs_np[i]}
    psa_params_list.append((samp, i))  # Store parameters and index


# Define a helper function to pass parameters and pop_size to run_sim
def run_sim_with_fixed_pop_size(params_and_idx):
    return run_sim(params_and_idx, pop_size=psa_pop_size)


# Use ThreadPoolExecutor for parallel execution
num_workers = 4  # Example: adjust based on your system (e.g., os.cpu_count())

psa_out = []
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    # `executor.map` applies run_sim_with_fixed_pop_size to each item in psa_params_list
    # and returns results in the order the tasks were submitted.
    for i, result in enumerate(
        executor.map(run_sim_with_fixed_pop_size, psa_params_list)
    ):
        dc, dq = result
        psa_out.append({"run": i, "dCost": dc, "dQALY": dq})
        if (i + 1) % 50 == 0:
            print(f"PSA Run {i + 1}/{psa_n} complete.")

psa_df = cudf.DataFrame(psa_out)
psa_df.to_csv("psa_samples.csv", index=False)
print("PSA results saved to psa_samples.csv")

# %% [markdown]
## 8. CE-Plane, CEAC & EVPI

print("\n--- Running Module 8: CE-Plane, CEAC & EVPI ---")
wtp = cp.linspace(0, 150000, 40)
dcost = psa_df["dCost"].to_cupy()
dqaly = psa_df["dQALY"].to_cupy()

# Filter out non-positive QALYs for stable NB calculation
valid_psa = dqaly > 0
nb = dqaly[valid_psa, None] * wtp[None, :] - dcost[valid_psa, None]

# CE-Plane scatter
plt.figure()
plt.scatter(dqaly.get(), dcost.get(), alpha=0.6)
plt.xlabel("Incremental QALYs (Delta QALY)")
plt.ylabel("Incremental Cost (Delta Cost)")
plt.title("Cost-Effectiveness Plane")
plt.grid(True)
plt.show()

# CEAC
prob = (nb > 0).mean(axis=0)
plt.figure()
plt.plot(wtp.get(), prob.get())
plt.title("Cost-Effectiveness Acceptability Curve (CEAC)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Probability Cost-Effective")
plt.ylim(0, 1)
plt.grid(True)
plt.show()

# EVPI - Corrected formula
evpi = cp.mean(cp.maximum(0, nb), axis=0) - cp.maximum(0, cp.mean(nb, axis=0))
plt.figure()
plt.plot(wtp.get(), evpi.get())
plt.title("Expected Value of Perfect Information (EVPI)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("EVPI")
plt.grid(True)
plt.show()

# %% [markdown]
## 9. One-Way Sensitivity (Tornado)

print("\n--- Running Module 9: One-Way Sensitivity (Tornado) ---")
tornado = []
base_params = {"familial_prop": 0.165, "yield_wgs": defaul_yield}
base_dc, base_dq = run_sim((base_params, 0), pop_size=10000)  # Pass dummy index
base_icer = base_dc / base_dq if base_dq != 0 else 0

for param in ["yield_wgs", "familial_prop"]:
    for change in [-0.2, 0.2]:
        p = base_params.copy()
        p[param] = p[param] * (1 + change)
        dc, dq = run_sim((p, 0), pop_size=10000)  # Pass dummy index
        icer = (dc / dq) if dq != 0 else 0
        tornado.append(
            {"param": param, "change": f"{int(change * 100)}%", "ICER": icer}
        )

tor_df = cudf.DataFrame(tornado)
labels = [
    f"{p} {c}"
    for p, c in zip(
        tor_df["param"].to_arrow().to_pylist(), tor_df["change"].to_arrow().to_pylist()
    )
]
icers = tor_df["ICER"].to_arrow().to_pylist()
bar_values = [(x - base_icer) for x in icers]
colors = ["red" if x < 0 else "green" for x in bar_values]

plt.figure()
plt.barh(labels, bar_values, color=colors)
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("Change in ICER from base (/QALY)")
plt.title("Tornado Sensitivity Analysis")
plt.tight_layout()
plt.show()

# %% [markdown]
## 10. Sub-group Analyses

print("\n--- Running Module 10: Sub-group Analyses ---")
sg = []

# 1. Severity
orig_sev_dist = SEVERITY_DIST.copy()
for sev, label in [(0, "Mild"), (1, "Moderate")]:
    SEVERITY_DIST = cp.array([1.0 if i == sev else 0.0 for i in range(2)])
    # Corrected: Pass params dictionary directly, remove tuple
    dc, dq = run_sim(
        {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=15000
    )
    sg.append({"group": f"Severity: {label}", "ICER": (dc / dq) if dq != 0 else 0})
SEVERITY_DIST = orig_sev_dist  # Restore

# 2. Age at test
for age in [0, 18, 30]:
    params = {"familial_prop": 0.165, "yield_wgs": defaul_yield, "min_test_age": age}
    # Corrected: Pass params dictionary directly, remove tuple
    dc, dq = run_sim(params, pop_size=15000)
    sg.append({"group": f"Min Test Age: {age}", "ICER": (dc / dq) if dq != 0 else 0})

# 3. Time-horizon
# Note: run_scenario still needs to be run sequentially as it collects detailed yearly data
(dc_full, dq_full), int_full = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=15000
)
dfp = int_full[int_full["arm"] == "prospective"]
dfr = int_full[int_full["arm"] == "retrospective"]
for yrs_n in [10, 20, len(YEARS)]:
    yr = START_YEAR + yrs_n - 1
    c1 = float(dfp.loc[dfp["year"] == yr, "costs"].values[0])
    c0 = float(dfr.loc[dfr["year"] == yr, "costs"].values[0])
    q1 = float(dfp.loc[dfp["year"] == yr, "qalys"].values[0])
    q0 = float(dfr.loc[dfr["year"] == yr, "qalys"].values[0])
    dc, dq = (c1 - c0), (q1 - q0)
    sg.append({"group": f"Horizon: {yrs_n} yrs", "ICER": (dc / dq) if dq != 0 else 0})

# Plot
sg_df = cudf.DataFrame(sg)
plt.figure()
plt.scatter(
    sg_df["ICER"].to_arrow().to_pylist(),
    sg_df["group"].to_arrow().to_pylist(),
    marker="o",
)
plt.axvline(
    base_icer, color="grey", linestyle="--", label=f"Base Case ICER ({base_icer:,.0f})"
)
plt.xlabel("ICER (/QALY)")
plt.title("Sub-group Analyses")
plt.legend()
plt.tight_layout()
plt.grid(axis="x")
plt.show()

# %% [markdown]
## 11. Cost-Mix Stacked Area (Prospective)

print("\n--- Running Module 11: Cost-Mix Stacked Area ---")
(_, _), mix = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
pm = mix[mix["arm"] == "prospective"]
cost_cols = [
    "cost_health",
    "cost_edu",
    "cost_respite",
    "cost_mobility",
    "cost_welfare",
    "cost_emp",
    "cost_morbidity",
]
years_host = pm["year"].to_arrow().to_pylist()
# Un-cumulate the data for stacking annual costs
data = [pm[col].to_arrow().to_pylist() for col in cost_cols]

plt.figure(figsize=(10, 6))
plt.stackplot(
    years_host, *data, labels=[c.replace("cost_", "").capitalize() for c in cost_cols]
)
plt.legend(loc="upper left")
plt.title("Cost Mix Over Time (Prospective Arm - Discounted Annual Costs)")
plt.xlabel("Year")
plt.ylabel("Annual Cost")
plt.show()

# %% [markdown]
## 12. 5-Year Budget Impact

print("\n--- Running Module 12: 5-Year Budget Impact ---")
budg = []
uptakes = [0.2, 0.5, 0.8]
yrs = YEARS[:5]
# Run simulation once to get the cost data
(_, _), df = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
dfp = df[df["arm"] == "prospective"]
dfr = df[df["arm"] == "retrospective"]

for u in uptakes:
    for yr in yrs:
        # Use 'annual_cost' which is not cumulative
        # Correct way to extract a single scalar from a cuDF Series
        c1 = float(dfp.loc[dfp["year"] == yr, "annual_cost"].to_cupy().item())
        c0 = float(dfr.loc[dfr["year"] == yr, "annual_cost"].to_cupy().item())

        # Budget impact is the incremental cost for the new intervention, scaled by uptake
        incremental_cost = c1 - c0
        budg.append({"uptake": u, "year": yr, "budget_impact": incremental_cost * u})

budg_df = cudf.DataFrame(budg)
plt.figure(figsize=(10, 6))
sns.lineplot(
    data=budg_df.to_pandas(), x="year", y="budget_impact", hue="uptake", marker="o"
)
plt.title("5-Year Budget Impact Analysis")
plt.xlabel("Year")
plt.ylabel("Budget Impact")
plt.grid(True)
plt.show()

# %% [markdown]
## 13. Equity Impacts (SES Quartiles)

print("\n--- Running Module 13: Equity Impacts (SES Quartiles) ---")
(_, _), eq_df = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
eq_df_p = eq_df[eq_df["arm"] == "prospective"]

plt.figure(figsize=(10, 6))
# Convert to pandas for easier plotting with seaborn's wide-form data handling
eq_df_p_pd = eq_df_p.to_pandas()
# Create a melted DataFrame for SES percentages
ses_cols = [f"ses_q{i}_pct" for i in range(5)]
equity_melted = eq_df_p_pd.melt(
    id_vars=["year"],
    value_vars=ses_cols,
    var_name="SES Quartile",
    value_name="Percentage of Population",
)
# Map column names for better labels
equity_melted["SES Quartile"] = (
    equity_melted["SES Quartile"]
    .str.replace("ses_q", "Quartile ")
    .str.replace("_pct", "")
)

sns.lineplot(
    data=equity_melted,
    x="year",
    y="Percentage of Population",
    hue="SES Quartile",
    marker="o",
)
plt.title("Distribution of Population by SES Quartile (Prospective Arm)")
plt.xlabel("Year")
plt.ylabel("Percentage of Population")
plt.grid(True)
plt.legend(title="SES Quartile")
plt.show()

# %% [markdown]
## 14. PSA Histograms

print("\n--- Running Module 14: PSA Histograms ---")
# Get data from the already computed psa_df
dcost_np = psa_df["dCost"].to_numpy()
dqaly_np = psa_df["dQALY"].to_numpy()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(dcost_np, kde=True, color="skyblue", bins=30)
plt.title("Histogram of Incremental Costs")
plt.xlabel("Incremental Cost")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
sns.histplot(dqaly_np, kde=True, color="lightcoral", bins=30)
plt.title("Histogram of Incremental QALYs")
plt.xlabel("Incremental QALYs")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

# %% [markdown]
## 15. Cost-Effectiveness Acceptability Frontier (CEAF)

print("\n--- Running Module 15: Cost-Effectiveness Acceptability Frontier (CEAF) ---")

# Assuming a range of WTP values
wtp_range = np.linspace(0, 150000, 100)  # Increased resolution for smoother curve

# Calculate Net Monetary Benefit (NMB) for each PSA sample at each WTP
# Convert CuPy arrays to NumPy for broadcsting with wtp_range (NumPy array)
# Or convert wtp_range to CuPy for full GPU, but for plotting, conversion back is needed anyway.
# Let's keep it on CPU for this plotting/analysis step as psa_df is already transferred.
dcost_cpu = psa_df["dCost"].to_numpy()
dqaly_cpu = psa_df["dQALY"].to_numpy()

# NMB = (delta QALY * WTP) - delta Cost
# Reshape dqaly and dcost for broadcasting
nmb = (dqaly_cpu[:, np.newaxis] * wtp_range[np.newaxis, :]) - dcost_cpu[:, np.newaxis]

# Calculate probability of being cost-effective at each WTP
prob_ce = np.mean(nmb > 0, axis=0)

# Identify the WTP values where prob_ce changes significantly
# For CEAF, we often look at the maximum NMB at each WTP value
# The CEAF shows the maximum expected Net Monetary Benefit for each WTP.
# This is typically derived from the average NMB over the PSA samples.
average_nmb = np.mean(nmb, axis=0)

# Plotting the CEAF (Expected Net Monetary Benefit)
plt.figure()
plt.plot(wtp_range, average_nmb)
plt.axhline(0, color="grey", linestyle="--", linewidth=0.8)
plt.title("Cost-Effectiveness Acceptability Frontier (CEAF)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Expected Net Monetary Benefit (ENMB)")
plt.grid(True)
plt.show()

# %% [markdown]
## 16. Input Parameter Summary

print("\n--- Running Module 16: Input Parameter Summary ---")

# Define all model parameters and their values
model_parameters = {
    "Simulation Settings": {
        "START_YEAR": START_YEAR,
        "END_YEAR": END_YEAR,
        "POP_SIZE": POP_SIZE,
        "DISCOUNT_RATE": DISCOUNT,
        "INFLATION_RATE": INFLATION_RATE,
        "COST_REFERENCE_YEAR": COST_REFERENCE_YEAR,
    },
    "Demographics & Health States": {
        "SEVERITY_DIST (Mild, Moderate)": SEVERITY_DIST.get().tolist(),  # .get() to bring to CPU
        "AGE_DIST": AGE_DIST.get().tolist(),  # .get() to bring to CPU
        "LIFE_EXPECTANCY (Mild, Moderate)": LIFE_EXP_CP.get().tolist(),
        "BASE_MORTALITY_RATES": ABS_MORT.get().tolist(),
        "MORTALITY_HR (Mild, Moderate)": MORT_HR.get().tolist(),
        "FAMILIAL_PROP (Base)": base_params["familial_prop"],
        "DIAGNOSIS_YIELD_WGS (Base)": defaul_yield,
    },
    "Utilities": {
        "U_BASE (Mild, Moderate)": U_BASE_CP.get().tolist(),
        "U_DECAY_RATE": U_DECAY,
        "U_DIAGNOSIS_GAIN": U_DIAG,
        "U_CARER_GAIN": U_CARER,
        "WGS_DIAG_QALY_ADJUSTMENT": WGS_DIAG_QALY_ADJUSTMENT,
    },
    "Costs (Adjusted to START_YEAR)": {
        "COST_INPATIENT (Mild, Moderate)": COST_INP_CP.get().tolist(),
        "COST_OUTPATIENT (Mild, Moderate)": COST_OUTP_CP.get().tolist(),
        "COST_PHARMACY (Mild, Moderate)": COST_PHARM_CP.get().tolist(),
        "COST_EDUCATION_Q0_SUPPORT (Low, Med, High)": EDU0.get().tolist(),
        "COST_EDUCATION_Q1_SUPPORT (Low, Med, High)": EDU1.get().tolist(),
        "COST_RESPITE_CARE": RESPITE,
        "COST_MOBILITY_SUPPORT": MOBILITY,
        "COST_WELFARE (Mild, Moderate)": WELFARE_CP.get().tolist(),
        "COST_EMPLOYMENT (Employed, Not in Labour Force, Unemployed)": EMP_CP.get().tolist(),
        "WGS_DIAG_COST_ADJUSTMENT": WGS_DIAG_COST_ADJUSTMENT,
    },
    "Morbidity": {
        "MORBIDITY_PROBABILITIES": MORB_PROB,
        "MORBIDITY_COSTS": MORB_COST_CP.get().tolist(),
        "MORBIDITY_QALYS_LOST": MORB_QALY_CP.get().tolist(),
    },
}

# Output to JSON
with open("model_parameters.json", "w") as f:
    json.dump(model_parameters, f, indent=4)

print("Model parameters summarized and saved to 'model_parameters.json'")

# Optionally, print to console as well for quick review
# for category, params in model_parameters.items():
#     print(f"\n--- {category} ---")
#     for param, value in params.items():
#         print(f"{param}: {value}")

# %% [markdown]
## 17. Publication Comparison (Placeholder)

print("\n--- Running Module 17: Publication Comparison ---")

# This module would typically involve comparing key outputs of your model
# (e.g., total costs, total QALYs, ICERs, prevalence rates)
# against published literature or real-world data.

# Example: Placeholder for comparing total QALYs to a hypothetical published range
# total_qaly_prospective = final_totals['prospective']['qaly'] # Requires final_totals from run_scenario, which isn't globally stored
# To make this robust, you'd need to explicitly run run_scenario or store its results
# more permanently. For this example, let's re-run for a single base case to get totals.
(_, _), base_scenario_detailed = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=POP_SIZE
)
base_total_qaly_p = float(
    base_scenario_detailed[base_scenario_detailed["arm"] == "prospective"][
        "qalys"
    ].iloc[-1]
)
base_total_cost_p = float(
    base_scenario_detailed[base_scenario_detailed["arm"] == "prospective"][
        "costs"
    ].iloc[-1]
)

print(f"Model's Total Prospective QALYs: {base_total_qaly_p:,.2f}")
print(f"Model's Total Prospective Costs: {base_total_cost_p:,.2f}")

published_qaly_range = (5000000, 6000000)  # Hypothetical range from literature
published_icer_range = (20000, 30000)  # Hypothetical range from literature

if published_qaly_range[0] <= base_total_qaly_p <= published_qaly_range[1]:
    print(
        f"Model's total QALYs ({base_total_qaly_p:,.2f}) are within published range {published_qaly_range}."
    )
else:
    print(
        f"Model's total QALYs ({base_total_qaly_p:,.2f}) are outside published range {published_qaly_range}."
    )

if published_icer_range[0] <= base_icer <= published_icer_range[1]:
    print(
        f"Model's base case ICER ({base_icer:,.0f}) is within published range {published_icer_range}."
    )
else:
    print(
        f"Model's base case ICER ({base_icer:,.0f}) is outside published range {published_icer_range}."
    )

print(
    "\nThis module serves as a placeholder for detailed validation against external data."
)

# %% [markdown]
## 18. Custom Plots/Visualizations

print("\n--- Running Module 18: Custom Plots/Visualizations ---")

# Example 1: Population Size Over Time
# We'd need to capture population size at each step to plot this.
# This requires modifying PopulationSystemGPU.step or run_scenario
# For demonstration, let's get it from the run_scenario data by counting
# distinct IDs or looking at the sum of annual QALYs/costs, but actual population
# count would be better. Let's add a population count to run_scenario's outputs.

# To properly plot population size over time, run_scenario needs to store `len(sim.pop)`
# at each step. This isn't currently captured in the `outputs` list.
# For a quick demo, we'll re-run a scenario and add pop count to its output.


def run_scenario_with_pop_count(params, pop_size=POP_SIZE):
    outputs = []
    for arm in ["retrospective", "prospective"]:
        sim = PopulationSystemGPU(pop_size, params["familial_prop"], arm)
        for yr in YEARS:
            sim.step(yr, params)
            current_pop_size = len(sim.pop)
            # Only store population count for simplicity here, can extend with other metrics
            outputs.append(
                {"arm": arm, "year": yr, "population_count": current_pop_size}
            )
    return cudf.DataFrame(outputs)


print("\nGenerating population trend plot...")
pop_trends_df = run_scenario_with_pop_count(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=POP_SIZE
)
pop_trends_pd = pop_trends_df.to_pandas()

plt.figure(figsize=(10, 6))
sns.lineplot(data=pop_trends_pd, x="year", y="population_count", hue="arm", marker="o")
plt.title("Population Size Over Time by Arm")
plt.xlabel("Year")
plt.ylabel("Population Count")
plt.grid(True)
plt.show()

# Example 2: Heatmap of Sensitivity to two parameters (if you had more comprehensive 2-way SA)
# This would require running a grid of simulations, similar to PSA but with fixed steps.
# For example:
# familial_range = np.linspace(0.1, 0.2, 5)
# yield_range = np.linspace(0.4, 0.6, 5)
# # You'd loop through these and store ICERs in a 2D array, then use sns.heatmap.
# print("\nPlaceholder for 2-way sensitivity heatmap.")

print("\nCustom visualizations complete.")

## v4a3

In [2]:
import warnings

import cupy as cp

warnings.filterwarnings("ignore")
cp.random.seed(42)

# %% [markdown]
## 2. Global Parameters & GPU Constants

START_YEAR, END_YEAR = 2020, 2100
YEARS = list(range(START_YEAR, END_YEAR + 1))
DISCOUNT = 0.035
POP_SIZE = 62000

# Costs are based on 2017 values from the paper, adjust to START_YEAR (2020)
COST_REFERENCE_YEAR = 2017
INFLATION_RATE = 0.03
INFLATION_FACTOR = (1 + INFLATION_RATE) ** (START_YEAR - COST_REFERENCE_YEAR)

# Demographics
SEVERITY_DIST = cp.array([0.70, 0.30])
AGE_BREAKS = cp.arange(0, 95, 5)
AGE_DIST = cp.array(
    [
        0.064,
        0.067,
        0.069,
        0.069,
        0.071,
        0.076,
        0.080,
        0.077,
        0.080,
        0.079,
        0.073,
        0.078,
        0.077,
        0.072,
        0.055,
        0.041,
        0.031,
        0.022,
        0.012,
    ]
)
AGE_DIST /= AGE_DIST.sum()

# Mortality
LIFE_EXP_CP = cp.array([74, 54])
ABS_MORT = cp.array(
    [
        0.0002,
        0.0001,
        0.0001,
        0.0002,
        0.0003,
        0.0005,
        0.0007,
        0.0011,
        0.0019,
        0.0035,
        0.0055,
        0.0085,
        0.0130,
        0.0200,
        0.0350,
        0.0600,
        0.0950,
        0.1500,
        0.2500,
    ]
)
MORT_HR = cp.array([2.0, 4.0])

defaul_yield = 0.45

# Fertility bands
FERT_AGE_SPEC = {
    (0, (15, 19)): 0.003,
    (0, (20, 24)): 0.008,
    (0, (25, 29)): 0.010,
    (0, (30, 34)): 0.007,
    (0, (35, 45)): 0.002,
    (1, (15, 19)): 0.001,
    (1, (20, 24)): 0.003,
    (1, (25, 29)): 0.004,
    (1, (30, 34)): 0.003,
    (1, (35, 45)): 0.001,
}
# Build GPU lookup table for fertility rates
fert_data = []
for (s, (lo, hi)), r in FERT_AGE_SPEC.items():
    for age in range(lo, hi + 1):
        fert_data.append({"severity": s, "age": age, "rate": r})
FERT_RATES_GPU = cudf.DataFrame(fert_data)

# Utilities
U_BASE_CP = cp.array([0.78, 0.65])
U_DECAY = 0.0005
U_DIAG = 0.02
U_CARER = 0.05

# Additional parameters for WGS diagnostic pathway impact
WGS_DIAG_COST_ADJUSTMENT = -1000  # Placeholder: e.g., representing annual cost savings from fewer diagnostic tests due to WGS
WGS_DIAG_QALY_ADJUSTMENT = 0.01  # Placeholder: e.g., representing additional annual QALY gain from earlier/precise intervention due to WGS

# Costs - Adjusted for inflation from COST_REFERENCE_YEAR to START_YEAR
COST_INP_CP = cp.array([5000, 15000]) * INFLATION_FACTOR
COST_OUTP_CP = cp.array([1000, 3000]) * INFLATION_FACTOR
COST_PHARM_CP = cp.array([200, 500]) * INFLATION_FACTOR
EDU0 = cp.array([16107, 27903, 46733]) * INFLATION_FACTOR
EDU1 = cp.array([18958, 30789, 49604]) * INFLATION_FACTOR
RESPITE = 13024 * INFLATION_FACTOR
MOBILITY = 2493 * INFLATION_FACTOR
WELFARE_CP = cp.array([3000, 8000]) * INFLATION_FACTOR
EMP_CP = cp.array([0, 10000, 0]) * INFLATION_FACTOR

# Morbidity
MORB_KEYS = ["epilepsy", "asd", "mental"]
MORB_PROB = {"epilepsy": 0.25, "asd": 0.30, "mental": 0.35}
MORB_COST_CP = cp.array([2000, 1500, 1000]) * INFLATION_FACTOR
MORB_QALY_CP = cp.array([0.04, 0.03, 0.05])

# %% [markdown]
## 3. Cohort Generator (GPU)


def sample_ages_gpu(n):
    idx = cp.random.choice(len(AGE_BREAKS), n, p=AGE_DIST)
    return (AGE_BREAKS[idx] + cp.random.randint(0, 5, n)).astype(int)


def gen_population_gpu(n, familial_prop, arm="prospective"):
    ages = sample_ages_gpu(n)
    sex = cp.random.choice(2, n)
    sev = cp.random.choice(2, n, p=SEVERITY_DIST)
    familial = cp.random.random(n) < familial_prop
    diag = (
        cp.zeros(n, bool)
        if arm == "prospective"
        else (cp.random.random(n) < defaul_yield)
    )
    living = cp.random.choice(4, n, p=cp.array([0.2, 0.1, 0.3, 0.4]))
    employment = cp.random.choice(3, n, p=cp.array([0.7, 0.2, 0.1]))
    uses_respite = cp.random.random(n) < 0.3
    needs_mobility = cp.random.random(n) < 0.2
    school = cp.where(ages < 5, -1, cp.where(ages < 12, 0, 1))
    support = cp.random.choice(3, n, p=cp.array([0.5, 0.3, 0.2]))
    ses = cp.random.choice(5, n)
    df = cudf.DataFrame(
        {
            "age": ages,
            "sex": sex,
            "severity": sev,
            "familial": familial,
            "diagnosed": diag,
            "living": living,
            "employment": employment,
            "uses_respite": uses_respite,
            "needs_mobility": needs_mobility,
            "school": school,
            "support": support,
            "ses": ses,
        }
    )
    # Add columns for outputs
    for col in [
        "annual_cost",
        "annual_qaly",
        "cost_health",
        "cost_edu",
        "cost_respite",
        "cost_mobility",
        "cost_welfare",
        "cost_emp",
        "cost_morbidity",
    ]:
        df[col] = 0.0
    return df

# %% [markdown]
## 4. Population System (GPU)


class PopulationSystemGPU:
    def __init__(self, pop_size, familial_prop, arm="prospective"):
        self.arm = arm
        self.pop = gen_population_gpu(pop_size, familial_prop, arm)

    def _mortality(self):
        df = self.pop
        ages = df["age"].to_cupy()
        sev = df["severity"].to_cupy()
        # Ensure age index is within bounds for ABS_MORT
        age_indices = cp.clip((ages // 5).astype(int), 0, len(ABS_MORT) - 1)
        surv = ABS_MORT[age_indices] * MORT_HR[sev]
        mask = (cp.random.random(len(df)) > surv) & (ages < LIFE_EXP_CP[sev])
        self.pop = df.iloc[cp.where(mask)[0]].reset_index(drop=True)

    def _fertility(self, familial_prop):
        df = self.pop
        moms = df[(df["sex"] == 0) & df["age"].between(15, 45)]
        if moms.empty:
            return
        mr = moms.merge(FERT_RATES_GPU, on=["severity", "age"], how="left").fillna(0)
        rates = mr["rate"].values
        births = cp.random.random(len(rates)) < rates
        idx = cp.nonzero(births)[0]
        if len(idx) > 0:
            babies = gen_population_gpu(len(idx), familial_prop, self.arm)
            babies["age"] = 0
            # Inherit 'familial' status from the mother
            flags = mr["familial"].iloc[idx.get()].reset_index(drop=True)
            babies["familial"] = flags
            self.pop = cudf.concat([df, babies], ignore_index=True)

    def _diagnosis(self, params):
        if self.arm != "prospective":
            return
        df = self.pop
        mask = (
            (~df["diagnosed"])
            & df["familial"]
            & (df["age"] >= params.get("min_test_age", 0))
        )
        cnt = int(mask.sum())
        if cnt > 0:
            # Use .loc with a boolean mask to assign values
            indices = df.index[mask]
            df.loc[indices, "diagnosed"] = cp.random.random(cnt) < params["yield_wgs"]
        self.pop = df

    def _cost_qaly(self, year, params):
        df = self.pop
        n = len(df)
        if n == 0:
            return

        disc = 1 / ((1 + DISCOUNT) ** (year - START_YEAR))
        sev = df["severity"].to_cupy()
        ages = df["age"].to_cupy()

        # Costs
        hc = (COST_INP_CP[sev] + COST_OUTP_CP[sev] + COST_PHARM_CP[sev]) * disc
        rc = df["uses_respite"].to_cupy().astype(int) * RESPITE * disc
        mc = (
            -df["needs_mobility"].to_cupy().astype(int) * MOBILITY * disc
        )  # Cost saving
        wc = WELFARE_CP[sev] * disc
        emp = EMP_CP[df["employment"].to_cupy()] * disc

        ec = cp.zeros(n, dtype=cp.float64)
        m0 = df["school"] == 0
        m1 = df["school"] == 1
        if not df.loc[m0].empty:
            ec[m0.to_cupy()] = EDU0[df.loc[m0, "support"].to_cupy()] * disc
        if not df.loc[m1].empty:
            ec[m1.to_cupy()] = EDU1[df.loc[m1, "support"].to_cupy()] * disc

        morb_cost = cp.zeros(n, dtype=cp.float64)
        base_rand = cp.random.random(n)
        for i, k in enumerate(MORB_KEYS):
            m = base_rand < MORB_PROB[k]
            morb_cost += m.astype(cp.float64) * MORB_COST_CP[i] * disc

        # Apply WGS-specific cost adjustment if in prospective arm and diagnosed
        if self.arm == "prospective":
            m = df["diagnosed"].to_cupy()
            morb_cost += m.astype(cp.float64) * WGS_DIAG_COST_ADJUSTMENT * disc

        # QALYs
        morb_qaly = cp.zeros(n, dtype=cp.float64)
        for i, k in enumerate(MORB_KEYS):
            m = base_rand < MORB_PROB[k]  # Use same random numbers for consistency
            morb_qaly += m.astype(cp.float64) * MORB_QALY_CP[i] * disc

        uq = U_BASE_CP[sev] * cp.exp(-U_DECAY * cp.clip(ages - 20, 0, None))
        total_qaly = (
            uq + df["diagnosed"].to_cupy().astype(int) * U_DIAG + U_CARER
        ) * disc - morb_qaly

        # Apply WGS-specific QALY adjustment if in prospective arm and diagnosed
        if self.arm == "prospective":
            m = df["diagnosed"].to_cupy()
            total_qaly += m.astype(cp.float64) * WGS_DIAG_QALY_ADJUSTMENT * disc

        # Assign back to DataFrame
        df["cost_health"] = hc
        df["cost_edu"] = ec
        df["cost_respite"] = rc
        df["cost_mobility"] = mc
        df["cost_welfare"] = wc
        df["cost_emp"] = emp
        df["cost_morbidity"] = morb_cost
        df["annual_cost"] = hc + ec + rc + mc + wc + emp + morb_cost
        df["annual_qaly"] = total_qaly
        self.pop = df

    def step(self, year, params):
        self._mortality()
        self._fertility(params["familial_prop"])
        self._diagnosis(params)
        self.pop["age"] += 1
        self._cost_qaly(year, params)

# %% [markdown]
## 5. Run Helpers


def run_sim(params, pop_size=POP_SIZE, seed_idx=None):
    """Fast runner for aggregate results (dCost, dQALY). Used for PSA, Tornado.
    Accepts an optional seed_idx for probabilistic runs."""
    if seed_idx is not None:
        cp.random.seed(42 + seed_idx)
    else:
        # Default seed for deterministic/non-PSA runs for reproducibility
        cp.random.seed(42)

    sim_r = PopulationSystemGPU(pop_size, params["familial_prop"], "retrospective")
    sim_p = PopulationSystemGPU(pop_size, params["familial_prop"], "prospective")

    cum_r_c = cum_r_q = cum_p_c = cum_p_q = 0.0

    for yr in YEARS:
        sim_r.step(yr, params)
        if not sim_r.pop.empty:
            cum_r_c += float(sim_r.pop["annual_cost"].sum())
            cum_r_q += float(sim_r.pop["annual_qaly"].sum())

        sim_p.step(yr, params)
        if not sim_p.pop.empty:
            cum_p_c += float(sim_p.pop["annual_cost"].sum())
            cum_p_q += float(sim_p.pop["annual_qaly"].sum())

    return (cum_p_c - cum_r_c, cum_p_q - cum_r_q)


def run_scenario(params, pop_size=POP_SIZE):
    """Detailed runner that returns aggregate results AND a per-year DataFrame."""
    outputs = []
    final_totals = {}

    for arm in ["retrospective", "prospective"]:
        sim = PopulationSystemGPU(pop_size, params["familial_prop"], arm)
        cum_c = 0.0
        cum_q = 0.0
        for yr in YEARS:
            sim.step(yr, params)
            df = sim.pop

            if df.empty:
                # Append a zero-row to maintain structure
                row_data = {
                    "arm": arm,
                    "year": yr,
                    "annual_cost": 0.0,
                    "cost_health": 0.0,
                    "cost_edu": 0.0,
                    "cost_respite": 0.0,
                    "cost_mobility": 0.0,
                    "cost_welfare": 0.0,
                    "cost_emp": 0.0,
                    "cost_morbidity": 0.0,
                    "costs": cum_c,
                    "qalys": cum_q,
                    **{f"ses_q{i}_pct": 0.0 for i in range(5)},
                }
                outputs.append(row_data)
                continue

            # Extract already-discounted annual components for detailed output
            hc = float(df["cost_health"].sum())
            edu = float(df["cost_edu"].sum())
            rc = float(df["cost_respite"].sum())
            mob = float(df["cost_mobility"].sum())
            wc = float(df["cost_welfare"].sum())
            emp = float(df["cost_emp"].sum())
            mc = float(df["cost_morbidity"].sum())

            annual_c_sum = float(df["annual_cost"].sum())
            annual_q_sum = float(df["annual_qaly"].sum())

            cum_c += annual_c_sum
            cum_q += annual_q_sum

            outputs.append(
                {
                    "arm": arm,
                    "year": yr,
                    "annual_cost": annual_c_sum,  # Crucial for budget impact
                    "cost_health": hc,
                    "cost_edu": edu,
                    "cost_respite": rc,
                    "cost_mobility": mob,
                    "cost_welfare": wc,
                    "cost_emp": emp,
                    "cost_morbidity": mc,
                    "costs": cum_c,
                    "qalys": cum_q,
                    **{
                        f"ses_q{i}_pct": float((df["ses"] == i).mean() * 100)
                        for i in range(5)
                    },
                }
            )

        final_totals[arm] = {"cost": cum_c, "qaly": cum_q}

    d_cost = final_totals["prospective"]["cost"] - final_totals["retrospective"]["cost"]
    d_qaly = final_totals["prospective"]["qaly"] - final_totals["retrospective"]["qaly"]

    return (d_cost, d_qaly), cudf.DataFrame(outputs)

# %% [markdown]
## 6. Deterministic Scenarios

print("--- Running Module 6: Deterministic Scenarios ---")
scenarios = {"low": 0.10, "base": 0.165, "high": 0.23}
results = []
for name, val in scenarios.items():
    # Calling run_sim without seed_idx
    dc, dq = run_sim({"familial_prop": val, "yield_wgs": defaul_yield}, pop_size=20000)
    icer = (dc / dq) if dq != 0 else 0
    results.append({"scenario": name, "dCost": dc, "dQALY": dq, "ICER": icer})

det_df = cudf.DataFrame(results)
det_df.to_csv("deterministic_summary.csv", index=False)
print(det_df)

plt.figure()
plt.bar(
    det_df["scenario"].to_arrow().to_pylist(), det_df["ICER"].to_arrow().to_pylist()
)
plt.title("Deterministic ICERs (/QALY)")
plt.ylabel("ICER (/QALY)")
plt.show()

# %% [markdown]
## 7. Probabilistic Sensitivity Analysis (PSA)

print("\n--- Running Module 7: Probabilistic Sensitivity Analysis ---")
psa_n = 300
psa_out = []

# Initialize Dask-CUDA cluster for parallel processing
cluster = LocalCUDACluster()
client = Client(cluster)

print(f"Dask dashboard link: {client.dashboard_link}")
print(f"Running {psa_n} PSA simulations in parallel using Dask-CUDA...")

# Prepare arguments for parallel map
psa_args = []
for i in range(psa_n):
    samp = {"familial_prop": beta.rvs(30, 150), "yield_wgs": beta.rvs(45, 55)}
    psa_args.append((samp, 8000, i))  # (params, pop_size, seed_idx)

# Map the run_sim function across the cluster
futures = client.map(lambda args: run_sim(args[0], args[1], args[2]), psa_args)
progress(futures)  # Show progress bar

# Gather results
results = client.gather(futures)

# Populate psa_out from gathered results
for i, (dc, dq) in enumerate(results):
    psa_out.append({"run": i, "dCost": dc, "dQALY": dq})

psa_df = cudf.DataFrame(psa_out)
psa_df.to_csv("psa_samples.csv", index=False)
print("PSA results saved to psa_samples.csv")

# Close the Dask client and cluster
client.close()
cluster.close()
print("Dask cluster closed.")

# %% [markdown]
## 8. CE-Plane, CEAC & EVPI

print("\n--- Running Module 8: CE-Plane, CEAC & EVPI ---")
wtp = cp.linspace(0, 150000, 40)
dcost = psa_df["dCost"].to_cupy()
dqaly = psa_df["dQALY"].to_cupy()

# Filter out non-positive QALYs for stable NB calculation
valid_psa = dqaly > 0
nb = dqaly[valid_psa, None] * wtp[None, :] - dcost[valid_psa, None]

# CE-Plane scatter
plt.figure()
plt.scatter(dqaly.get(), dcost.get(), alpha=0.6)
plt.xlabel("Incremental QALYs (Delta QALY)")
plt.ylabel("Incremental Cost (Delta Cost)")
plt.title("Cost-Effectiveness Plane")
plt.grid(True)
plt.show()

# CEAC
prob = (nb > 0).mean(axis=0)
plt.figure()
plt.plot(wtp.get(), prob.get())
plt.title("Cost-Effectiveness Acceptability Curve (CEAC)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Probability Cost-Effective")
plt.ylim(0, 1)
plt.grid(True)
plt.show()

# EVPI - Corrected formula
evpi = cp.mean(cp.maximum(0, nb), axis=0) - cp.maximum(0, cp.mean(nb, axis=0))
plt.figure()
plt.plot(wtp.get(), evpi.get())
plt.title("Expected Value of Perfect Information (EVPI)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("EVPI")
plt.grid(True)
plt.show()

# %% [markdown]
## 9. One-Way Sensitivity (Tornado)

print("\n--- Running Module 9: One-Way Sensitivity (Tornado) ---")
tornado = []
base_params = {"familial_prop": 0.165, "yield_wgs": defaul_yield}
# Calling run_sim without seed_idx
base_dc, base_dq = run_sim(base_params, pop_size=10000)
base_icer = base_dc / base_dq if base_dq != 0 else 0

for param in ["yield_wgs", "familial_prop"]:
    for change in [-0.2, 0.2]:
        p = base_params.copy()
        p[param] = p[param] * (1 + change)
        # Calling run_sim without seed_idx
        dc, dq = run_sim(p, pop_size=10000)
        icer = (dc / dq) if dq != 0 else 0
        tornado.append(
            {"param": param, "change": f"{int(change * 100)}%", "ICER": icer}
        )

tor_df = cudf.DataFrame(tornado)
labels = [
    f"{p} {c}"
    for p, c in zip(
        tor_df["param"].to_arrow().to_pylist(), tor_df["change"].to_arrow().to_pylist()
    )
]
icers = tor_df["ICER"].to_arrow().to_pylist()
bar_values = [(x - base_icer) for x in icers]
colors = ["red" if x < 0 else "green" for x in bar_values]

plt.figure()
plt.barh(labels, bar_values, color=colors)
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("Change in ICER from base (/QALY)")
plt.title("Tornado Sensitivity Analysis")
plt.tight_layout()
plt.show()

# %% [markdown]
## 10. Sub-group Analyses

print("\n--- Running Module 10: Sub-group Analyses ---")
sg = []

# 1. Severity
orig_sev_dist = SEVERITY_DIST.copy()
for sev, label in [(0, "Mild"), (1, "Moderate")]:
    SEVERITY_DIST = cp.array([1.0 if i == sev else 0.0 for i in range(2)])
    # Calling run_sim without seed_idx
    dc, dq = run_sim(
        {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=15000
    )
    sg.append({"group": f"Severity: {label}", "ICER": (dc / dq) if dq != 0 else 0})
SEVERITY_DIST = orig_sev_dist  # Restore

# 2. Age at test
for age in [0, 18, 30]:
    params = {"familial_prop": 0.165, "yield_wgs": defaul_yield, "min_test_age": age}
    # Calling run_sim without seed_idx
    dc, dq = run_sim(params, pop_size=15000)
    sg.append({"group": f"Min Test Age: {age}", "ICER": (dc / dq) if dq != 0 else 0})

# 3. Time-horizon
# Note: run_scenario still needs to be run sequentially as it collects detailed yearly data
(dc_full, dq_full), int_full = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=15000
)
dfp = int_full[int_full["arm"] == "prospective"]
dfr = int_full[int_full["arm"] == "retrospective"]
for yrs_n in [10, 20, len(YEARS)]:
    yr = START_YEAR + yrs_n - 1
    c1 = float(dfp.loc[dfp["year"] == yr, "costs"].values[0])
    c0 = float(dfr.loc[dfr["year"] == yr, "costs"].values[0])
    q1 = float(dfp.loc[dfp["year"] == yr, "qalys"].values[0])
    q0 = float(dfr.loc[dfr["year"] == yr, "qalys"].values[0])
    dc, dq = (c1 - c0), (q1 - q0)
    sg.append({"group": f"Horizon: {yrs_n} yrs", "ICER": (dc / dq) if dq != 0 else 0})

# Plot
sg_df = cudf.DataFrame(sg)
plt.figure()
plt.scatter(
    sg_df["ICER"].to_arrow().to_pylist(),
    sg_df["group"].to_arrow().to_pylist(),
    marker="o",
)
plt.axvline(
    base_icer, color="grey", linestyle="--", label=f"Base Case ICER ({base_icer:,.0f})"
)
plt.xlabel("ICER (/QALY)")
plt.title("Sub-group Analyses")
plt.legend()
plt.tight_layout()
plt.grid(axis="x")
plt.show()

# %% [markdown]
## 11. Cost-Mix Stacked Area (Prospective)

print("\n--- Running Module 11: Cost-Mix Stacked Area ---")
(_, _), mix = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
pm = mix[mix["arm"] == "prospective"]
cost_cols = [
    "cost_health",
    "cost_edu",
    "cost_respite",
    "cost_mobility",
    "cost_welfare",
    "cost_emp",
    "cost_morbidity",
]
years_host = pm["year"].to_arrow().to_pylist()
# Un-cumulate the data for stacking annual costs
data = [pm[col].to_arrow().to_pylist() for col in cost_cols]

plt.figure(figsize=(10, 6))
plt.stackplot(
    years_host, *data, labels=[c.replace("cost_", "").capitalize() for c in cost_cols]
)
plt.legend(loc="upper left")
plt.title("Cost Mix Over Time (Prospective Arm - Discounted Annual Costs)")
plt.xlabel("Year")
plt.ylabel("Annual Cost")
plt.show()

# %% [markdown]
## 12. 5-Year Budget Impact

print("\n--- Running Module 12: 5-Year Budget Impact ---")
budg = []
uptakes = [0.2, 0.5, 0.8]
yrs = YEARS[:5]
# Run simulation once to get the cost data
(_, _), df = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
dfp = df[df["arm"] == "prospective"]
dfr = df[df["arm"] == "retrospective"]

for u in uptakes:
    for yr in yrs:
        # Use 'annual_cost' which is not cumulative
        # Correct way to extract a single scalar from a cuDF Series
        c1 = float(dfp.loc[dfp["year"] == yr, "annual_cost"].to_cupy().item())
        c0 = float(dfr.loc[dfr["year"] == yr, "annual_cost"].to_cupy().item())

        # Budget impact is the incremental cost for the new intervention, scaled by uptake
        incremental_cost = c1 - c0
        budg.append({"uptake": u, "year": yr, "budget_impact": incremental_cost * u})

budg_df = cudf.DataFrame(budg)
plt.figure()
for u in uptakes:
    sub = budg_df[budg_df["uptake"] == u]
    yrs_h = sub["year"].to_arrow().to_pylist()
    vals = sub["budget_impact"].to_arrow().to_pylist()
    plt.plot(yrs_h, vals, label=f"Uptake {int(u * 100)}%", marker="o")
plt.title("5-Year Budget Impact")
plt.xlabel("Year")
plt.ylabel("Incremental Budget Impact")
plt.legend()
plt.grid(True)
plt.show()

# %% [markdown]
## 13. Equity by SES Quintile

print("\n--- Running Module 13: Equity by SES Quintile ---")
(_, _), eq_int = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
bas = eq_int[eq_int["arm"] == "prospective"]
# Select the last year of the simulation
row_series = bas[bas["year"] == YEARS[-1]].iloc[0]

ses_cols = [f"ses_q{i}_pct" for i in range(5)]
# Extract scalar values from the series by converting to cupy and then using .item()
ses_vals = [float(row_series[col].to_cupy().item()) for col in ses_cols]
labels = [f"Quintile {i + 1}" for i in range(5)]

plt.figure()
plt.bar(labels, ses_vals)
plt.title(f"SES Distribution of Population in Final Year ({YEARS[-1]})")
plt.xlabel("SES Quintile")
plt.ylabel("Percentage of Population (%)")
plt.tight_layout()
plt.show()

# %% [markdown]
## 14. PSA Histograms

print("\n--- Running Module 14: PSA Histograms ---")
cost_list = psa_df["dCost"].to_arrow().to_pylist()
qaly_list = psa_df["dQALY"].to_arrow().to_pylist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.hist(cost_list, bins=30, color="skyblue", edgecolor="black")
ax1.set_title("PSA Distribution of Incremental Cost")
ax1.set_xlabel("Delta Cost")
ax1.set_ylabel("Frequency")

ax2.hist(qaly_list, bins=30, color="salmon", edgecolor="black")
ax2.set_title("PSA Distribution of Incremental QALYs")
ax2.set_xlabel("Delta QALY")
ax2.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# %% [markdown]
## 15. CEAF (Cost-Effectiveness Acceptability Frontier)

print("\n--- Running Module 15: CEAF ---")
# Reuse 'prob' (CEAC probabilities) from Module 8
prob_host = prob.get()  # Get the CuPy array to host memory (NumPy array)
ceaf = np.maximum.accumulate(prob_host)  # Use numpy for the accumulate operation

plt.figure()
plt.plot(wtp.get(), prob.get(), label="CEAC")
plt.plot(wtp.get(), ceaf, label="CEAF", linestyle="--")  # ceaf is already a NumPy array
plt.title("CEAC and CEAF")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Probability")
plt.legend()
plt.grid(True)
plt.show()

# %% [markdown]
## 16. Input Parameter Summary

print("\n--- Running Module 16: Saving Input Parameter Summary ---")
params_summary = {
    "DISCOUNT": DISCOUNT,
    "default_yield": defaul_yield,
    "U_DIAG": U_DIAG,
    "U_CARER": U_CARER,
    "RESPITE": RESPITE,
    "MOBILITY": MOBILITY,
    "FERT_AGE_SPEC": json.dumps(
        {f"{k[0]}_{k[1][0]}_{k[1][1]}": v for k, v in FERT_AGE_SPEC.items()}
    ),
    "MORB_PROB": json.dumps(MORB_PROB),
    "AGE_DIST": json.dumps(AGE_DIST.get().tolist()),
    "COST_REFERENCE_YEAR": COST_REFERENCE_YEAR,
    "INFLATION_RATE": INFLATION_RATE,
    "INFLATION_FACTOR": float(INFLATION_FACTOR),
}

param_df = cudf.DataFrame(
    {
        "parameter": list(params_summary.keys()),
        "value": [str(v) for v in params_summary.values()],
    }
)

param_df.to_csv("input_summary.csv", index=False)
print("Input parameters saved to input_summary.csv")

# %% [markdown]
## 17. Publication Comparison

print("\n--- Running Module 17: Publication Comparison ---")
# Published point estimates for comparison
published = {"base": {"dCost": 480000, "dQALY": 6.5, "ICER": 73846}}

pub_rows = []
for scen, vals in published.items():
    pub_rows.append(
        {
            "scenario": scen,
            "dCost_pub": vals["dCost"],
            "dQALY_pub": vals["dQALY"],
            "ICER_pub": vals["ICER"],
        }
    )
pub_df = cudf.DataFrame(pub_rows)

# Merge with your deterministic simulation output (det_df from Module 6)
df_cmp = det_df.merge(pub_df, on="scenario", how="left")
df_cmp = df_cmp.rename(
    columns={"dCost": "dCost_sim", "dQALY": "dQALY_sim", "ICER": "ICER_sim"}
)

print("Comparison of Simulation vs. Published Data:")
print(df_cmp.to_pandas().to_string())  # Use pandas for cleaner console printing
df_cmp.to_csv("publication_comparison.csv", index=False)
print("Saved comparison to publication_comparison.csv")

# %% [markdown]
## 18. Familial ID Prevalence Sensitivity Analysis

print("\n--- Running Module 18: Familial ID Prevalence Sensitivity Analysis ---")
familial_prevalence_range = cp.linspace(0.10, 0.23, 10).get()  # Range from 10% to 23%
familial_sens_results = []
for f_prop in familial_prevalence_range:
    # Calling run_sim without seed_idx
    dc, dq = run_sim(
        {"familial_prop": f_prop, "yield_wgs": defaul_yield}, pop_size=10000
    )
    icer = (dc / dq) if dq != 0 else 0
    familial_sens_results.append(
        {"familial_prop": f_prop, "dCost": dc, "dQALY": dq, "ICER": icer}
    )

familial_sens_df = cudf.DataFrame(familial_sens_results)
print("Familial ID Prevalence Sensitivity Results:")
print(familial_sens_df.to_pandas().to_string())

plt.figure(figsize=(8, 6))
plt.plot(
    familial_sens_df["familial_prop"].to_arrow().to_pylist(),
    familial_sens_df["ICER"].to_arrow().to_pylist(),
    marker="o",
    linestyle="-",
)
plt.xlabel("Familial ID Prevalence")
plt.ylabel("ICER (/QALY)")
plt.title("Sensitivity to Familial ID Prevalence")
plt.grid(True)
plt.show()

## v4a4

In [23]:
import warnings

import cupy as cp

warnings.filterwarnings("ignore")
cp.random.seed(42)

# %% [markdown]
## 2. Global Parameters & GPU Constants

START_YEAR, END_YEAR = 2020, 2100
YEARS = list(range(START_YEAR, END_YEAR + 1))
DISCOUNT = 0.035
POP_SIZE = 62000

# Costs are based on 2017 values from the paper, adjust to START_YEAR (2020)
COST_REFERENCE_YEAR = 2017
INFLATION_RATE = 0.03
INFLATION_FACTOR = (1 + INFLATION_RATE) ** (START_YEAR - COST_REFERENCE_YEAR)

# Demographics
SEVERITY_DIST = cp.array([0.70, 0.30])
AGE_BREAKS = cp.arange(0, 95, 5)
AGE_DIST = cp.array(
    [
        0.064,
        0.067,
        0.069,
        0.069,
        0.071,
        0.076,
        0.080,
        0.077,
        0.080,
        0.079,
        0.073,
        0.078,
        0.077,
        0.072,
        0.055,
        0.041,
        0.031,
        0.022,
        0.012,
    ]
)
AGE_DIST /= AGE_DIST.sum()

# Mortality
LIFE_EXP_CP = cp.array([74, 54])
ABS_MORT = cp.array(
    [
        0.0002,
        0.0001,
        0.0001,
        0.0002,
        0.0003,
        0.0005,
        0.0007,
        0.0011,
        0.0019,
        0.0035,
        0.0055,
        0.0085,
        0.0130,
        0.0200,
        0.0350,
        0.0600,
        0.0950,
        0.1500,
        0.2500,
    ]
)
MORT_HR = cp.array([2.0, 4.0])

defaul_yield = 0.45

# Fertility bands
FERT_AGE_SPEC = {
    (0, (15, 19)): 0.003,
    (0, (20, 24)): 0.008,
    (0, (25, 29)): 0.010,
    (0, (30, 34)): 0.007,
    (0, (35, 45)): 0.002,
    (1, (15, 19)): 0.001,
    (1, (20, 24)): 0.003,
    (1, (25, 29)): 0.004,
    (1, (30, 34)): 0.003,
    (1, (35, 45)): 0.001,
}
# Build GPU lookup table for fertility rates
fert_data = []
for (s, (lo, hi)), r in FERT_AGE_SPEC.items():
    for age in range(lo, hi + 1):
        fert_data.append({"severity": s, "age": age, "rate": r})
FERT_RATES_GPU = cudf.DataFrame(fert_data)

# Utilities
U_BASE_CP = cp.array([0.78, 0.65])
U_DECAY = 0.0005
U_DIAG = 0.02
U_CARER = 0.05

# Additional parameters for WGS diagnostic pathway impact
WGS_DIAG_COST_ADJUSTMENT = -1000  # Placeholder: e.g., representing annual cost savings from fewer diagnostic tests due to WGS
WGS_DIAG_QALY_ADJUSTMENT = 0.01  # Placeholder: e.g., representing additional annual QALY gain from earlier/precise intervention due to WGS

# Costs - Adjusted for inflation from COST_REFERENCE_YEAR to START_YEAR
COST_INP_CP = cp.array([5000, 15000]) * INFLATION_FACTOR
COST_OUTP_CP = cp.array([1000, 3000]) * INFLATION_FACTOR
COST_PHARM_CP = cp.array([200, 500]) * INFLATION_FACTOR
EDU0 = cp.array([16107, 27903, 46733]) * INFLATION_FACTOR
EDU1 = cp.array([18958, 30789, 49604]) * INFLATION_FACTOR
RESPITE = 13024 * INFLATION_FACTOR
MOBILITY = 2493 * INFLATION_FACTOR
WELFARE_CP = cp.array([3000, 8000]) * INFLATION_FACTOR
EMP_CP = cp.array([0, 10000, 0]) * INFLATION_FACTOR

# Morbidity
MORB_KEYS = ["epilepsy", "asd", "mental"]
MORB_PROB = {"epilepsy": 0.25, "asd": 0.30, "mental": 0.35}
MORB_COST_CP = cp.array([2000, 1500, 1000]) * INFLATION_FACTOR
MORB_QALY_CP = cp.array([0.04, 0.03, 0.05])

# %% [markdown]
## 3. Cohort Generator (GPU)


def sample_ages_gpu(n):
    idx = cp.random.choice(len(AGE_BREAKS), n, p=AGE_DIST)
    return (AGE_BREAKS[idx] + cp.random.randint(0, 5, n)).astype(int)


def gen_population_gpu(n, familial_prop, arm="prospective"):
    ages = sample_ages_gpu(n)
    sex = cp.random.choice(2, n)
    sev = cp.random.choice(2, n, p=SEVERITY_DIST)
    familial = cp.random.random(n) < familial_prop
    diag = (
        cp.zeros(n, bool)
        if arm == "prospective"
        else (cp.random.random(n) < defaul_yield)
    )
    living = cp.random.choice(4, n, p=cp.array([0.2, 0.1, 0.3, 0.4]))
    employment = cp.random.choice(3, n, p=cp.array([0.7, 0.2, 0.1]))
    uses_respite = cp.random.random(n) < 0.3
    needs_mobility = cp.random.random(n) < 0.2
    school = cp.where(ages < 5, -1, cp.where(ages < 12, 0, 1))
    support = cp.random.choice(3, n, p=cp.array([0.5, 0.3, 0.2]))
    ses = cp.random.choice(5, n)
    df = cudf.DataFrame(
        {
            "age": ages,
            "sex": sex,
            "severity": sev,
            "familial": familial,
            "diagnosed": diag,
            "living": living,
            "employment": employment,
            "uses_respite": uses_respite,
            "needs_mobility": needs_mobility,
            "school": school,
            "support": support,
            "ses": ses,
        }
    )
    # Add columns for outputs
    for col in [
        "annual_cost",
        "annual_qaly",
        "cost_health",
        "cost_edu",
        "cost_respite",
        "cost_mobility",
        "cost_welfare",
        "cost_emp",
        "cost_morbidity",
    ]:
        df[col] = 0.0
    return df

# %% [markdown]
## 4. Population System (GPU)


class PopulationSystemGPU:
    def __init__(self, pop_size, familial_prop, arm="prospective"):
        self.arm = arm
        self.pop = gen_population_gpu(pop_size, familial_prop, arm)

    def _mortality(self):
        df = self.pop
        ages = df["age"].to_cupy()
        sev = df["severity"].to_cupy()
        # Ensure age index is within bounds for ABS_MORT
        age_indices = cp.clip((ages // 5).astype(int), 0, len(ABS_MORT) - 1)
        surv = ABS_MORT[age_indices] * MORT_HR[sev]
        mask = (cp.random.random(len(df)) > surv) & (ages < LIFE_EXP_CP[sev])
        self.pop = df.iloc[cp.where(mask)[0]].reset_index(drop=True)

    def _fertility(self, familial_prop):
        df = self.pop
        moms = df[(df["sex"] == 0) & df["age"].between(15, 45)]
        if moms.empty:
            return
        mr = moms.merge(FERT_RATES_GPU, on=["severity", "age"], how="left").fillna(0)
        rates = mr["rate"].values
        births = cp.random.random(len(rates)) < rates
        idx = cp.nonzero(births)[0]
        if len(idx) > 0:
            babies = gen_population_gpu(len(idx), familial_prop, self.arm)
            babies["age"] = 0
            # Inherit 'familial' status from the mother
            flags = mr["familial"].iloc[idx.get()].reset_index(drop=True)
            babies["familial"] = flags
            self.pop = cudf.concat([df, babies], ignore_index=True)

    def _diagnosis(self, params):
        if self.arm != "prospective":
            return
        df = self.pop
        mask = (
            (~df["diagnosed"])
            & df["familial"]
            & (df["age"] >= params.get("min_test_age", 0))
        )
        cnt = int(mask.sum())
        if cnt > 0:
            # Use .loc with a boolean mask to assign values
            indices = df.index[mask]
            df.loc[indices, "diagnosed"] = cp.random.random(cnt) < params["yield_wgs"]
        self.pop = df

    def _cost_qaly(self, year, params):
        df = self.pop
        n = len(df)
        if n == 0:
            return

        disc = 1 / ((1 + DISCOUNT) ** (year - START_YEAR))
        sev = df["severity"].to_cupy()
        ages = df["age"].to_cupy()

        # Costs
        hc = (COST_INP_CP[sev] + COST_OUTP_CP[sev] + COST_PHARM_CP[sev]) * disc
        rc = df["uses_respite"].to_cupy().astype(int) * RESPITE * disc
        mc = (
            -df["needs_mobility"].to_cupy().astype(int) * MOBILITY * disc
        )  # Cost saving
        wc = WELFARE_CP[sev] * disc
        emp = EMP_CP[df["employment"].to_cupy()] * disc

        ec = cp.zeros(n, dtype=cp.float64)
        m0 = df["school"] == 0
        m1 = df["school"] == 1
        if not df.loc[m0].empty:
            ec[m0.to_cupy()] = EDU0[df.loc[m0, "support"].to_cupy()] * disc
        if not df.loc[m1].empty:
            ec[m1.to_cupy()] = EDU1[df.loc[m1, "support"].to_cupy()] * disc

        morb_cost = cp.zeros(n, dtype=cp.float64)
        base_rand = cp.random.random(n)
        for i, k in enumerate(MORB_KEYS):
            m = base_rand < MORB_PROB[k]
            morb_cost += m.astype(cp.float64) * MORB_COST_CP[i] * disc

        # Apply WGS-specific cost adjustment if in prospective arm and diagnosed
        if self.arm == "prospective":
            m = df["diagnosed"].to_cupy()
            morb_cost += m.astype(cp.float64) * WGS_DIAG_COST_ADJUSTMENT * disc

        # QALYs
        morb_qaly = cp.zeros(n, dtype=cp.float64)
        for i, k in enumerate(MORB_KEYS):
            m = base_rand < MORB_PROB[k]  # Use same random numbers for consistency
            morb_qaly += m.astype(cp.float64) * MORB_QALY_CP[i] * disc

        uq = U_BASE_CP[sev] * cp.exp(-U_DECAY * cp.clip(ages - 20, 0, None))
        total_qaly = (
            uq + df["diagnosed"].to_cupy().astype(int) * U_DIAG + U_CARER
        ) * disc - morb_qaly

        # Apply WGS-specific QALY adjustment if in prospective arm and diagnosed
        if self.arm == "prospective":
            m = df["diagnosed"].to_cupy()
            total_qaly += m.astype(cp.float64) * WGS_DIAG_QALY_ADJUSTMENT * disc

        # Assign back to DataFrame
        df["cost_health"] = hc
        df["cost_edu"] = ec
        df["cost_respite"] = rc
        df["cost_mobility"] = mc
        df["cost_welfare"] = wc
        df["cost_emp"] = emp
        df["cost_morbidity"] = morb_cost
        df["annual_cost"] = hc + ec + rc + mc + wc + emp + morb_cost
        df["annual_qaly"] = total_qaly
        self.pop = df

    def step(self, year, params):
        self._mortality()
        self._fertility(params["familial_prop"])
        self._diagnosis(params)
        self.pop["age"] += 1
        self._cost_qaly(year, params)

# %% [markdown]
## 5. Run Helpers


def run_sim(params, pop_size=POP_SIZE, seed_idx=None):
    """Fast runner for aggregate results (dCost, dQALY). Used for PSA, Tornado.
    Accepts an optional seed_idx for probabilistic runs."""
    if seed_idx is not None:
        cp.random.seed(42 + seed_idx)
    else:
        # Default seed for deterministic/non-PSA runs for reproducibility
        cp.random.seed(42)

    sim_r = PopulationSystemGPU(pop_size, params["familial_prop"], "retrospective")
    sim_p = PopulationSystemGPU(pop_size, params["familial_prop"], "prospective")

    cum_r_c = cum_r_q = cum_p_c = cum_p_q = 0.0

    for yr in YEARS:
        sim_r.step(yr, params)
        if not sim_r.pop.empty:
            cum_r_c += float(sim_r.pop["annual_cost"].sum())
            cum_r_q += float(sim_r.pop["annual_qaly"].sum())

        sim_p.step(yr, params)
        if not sim_p.pop.empty:
            cum_p_c += float(sim_p.pop["annual_cost"].sum())
            cum_p_q += float(sim_p.pop["annual_qaly"].sum())

    return (cum_p_c - cum_r_c, cum_p_q - cum_r_q)


def run_scenario(params, pop_size=POP_SIZE):
    """Detailed runner that returns aggregate results AND a per-year DataFrame."""
    outputs = []
    final_totals = {}

    for arm in ["retrospective", "prospective"]:
        sim = PopulationSystemGPU(pop_size, params["familial_prop"], arm)
        cum_c = 0.0
        cum_q = 0.0
        for yr in YEARS:
            sim.step(yr, params)
            df = sim.pop

            if df.empty:
                # Append a zero-row to maintain structure
                row_data = {
                    "arm": arm,
                    "year": yr,
                    "annual_cost": 0.0,
                    "cost_health": 0.0,
                    "cost_edu": 0.0,
                    "cost_respite": 0.0,
                    "cost_mobility": 0.0,
                    "cost_welfare": 0.0,
                    "cost_emp": 0.0,
                    "cost_morbidity": 0.0,
                    "costs": cum_c,
                    "qalys": cum_q,
                    **{f"ses_q{i}_pct": 0.0 for i in range(5)},
                }
                outputs.append(row_data)
                continue

            # Extract already-discounted annual components for detailed output
            hc = float(df["cost_health"].sum())
            edu = float(df["cost_edu"].sum())
            rc = float(df["cost_respite"].sum())
            mob = float(df["cost_mobility"].sum())
            wc = float(df["cost_welfare"].sum())
            emp = float(df["cost_emp"].sum())
            mc = float(df["cost_morbidity"].sum())

            annual_c_sum = float(df["annual_cost"].sum())
            annual_q_sum = float(df["annual_qaly"].sum())

            cum_c += annual_c_sum
            cum_q += annual_q_sum

            outputs.append(
                {
                    "arm": arm,
                    "year": yr,
                    "annual_cost": annual_c_sum,  # Crucial for budget impact
                    "cost_health": hc,
                    "cost_edu": edu,
                    "cost_respite": rc,
                    "cost_mobility": mob,
                    "cost_welfare": wc,
                    "cost_emp": emp,
                    "cost_morbidity": mc,
                    "costs": cum_c,
                    "qalys": cum_q,
                    **{
                        f"ses_q{i}_pct": float((df["ses"] == i).mean() * 100)
                        for i in range(5)
                    },
                }
            )

        final_totals[arm] = {"cost": cum_c, "qaly": cum_q}

    d_cost = final_totals["prospective"]["cost"] - final_totals["retrospective"]["cost"]
    d_qaly = final_totals["prospective"]["qaly"] - final_totals["retrospective"]["qaly"]

    return (d_cost, d_qaly), cudf.DataFrame(outputs)

# %% [markdown]
## 6. Deterministic Scenarios

print("--- Running Module 6: Deterministic Scenarios ---")
scenarios = {"low": 0.10, "base": 0.165, "high": 0.23}
results = []
for name, val in scenarios.items():
    # Calling run_sim without seed_idx
    dc, dq = run_sim({"familial_prop": val, "yield_wgs": defaul_yield}, pop_size=20000)
    icer = (dc / dq) if dq != 0 else 0
    results.append({"scenario": name, "dCost": dc, "dQALY": dq, "ICER": icer})

det_df = cudf.DataFrame(results)
det_df.to_csv("deterministic_summary.csv", index=False)
print(det_df)

plt.figure()
plt.bar(
    det_df["scenario"].to_arrow().to_pylist(), det_df["ICER"].to_arrow().to_pylist()
)
plt.title("Deterministic ICERs (/QALY)")
plt.ylabel("ICER (/QALY)")
plt.show()

# %% [markdown]
## 7. Probabilistic Sensitivity Analysis (PSA)

print("\n--- Running Module 7: Probabilistic Sensitivity Analysis ---")
psa_n = 300
psa_out = []

# Initialize Dask-CUDA cluster for parallel processing
cluster = LocalCUDACluster()
client = Client(cluster)

print(f"Dask dashboard link: {client.dashboard_link}")
print(f"Running {psa_n} PSA simulations in parallel using Dask-CUDA...")

# Prepare arguments for parallel map
psa_args = []
for i in range(psa_n):
    samp = {"familial_prop": beta.rvs(30, 150), "yield_wgs": beta.rvs(45, 55)}
    # Store the sampled parameters along with the run arguments
    psa_args.append((samp, 8000, i))  # (params, pop_size, seed_idx)

# Map the run_sim function across the cluster
futures = client.map(lambda args: run_sim(args[0], args[1], args[2]), psa_args)
progress(futures)  # Show progress bar

# Gather results
results = client.gather(futures)

# Populate psa_out from gathered results, including sampled parameters
for i, (dc, dq) in enumerate(results):
    # Retrieve the original sampled parameters for this run from psa_args
    original_samp = psa_args[i][0]
    psa_out.append(
        {
            "run": i,
            "dCost": dc,
            "dQALY": dq,
            "familial_prop": original_samp["familial_prop"],  # Store sampled value
            "yield_wgs": original_samp["yield_wgs"],  # Store sampled value
        }
    )

psa_df = cudf.DataFrame(psa_out)
psa_df.to_csv("psa_samples.csv", index=False)
print("PSA results saved to psa_samples.csv")

# Close the Dask client and cluster
client.close()
cluster.close()
print("Dask cluster closed.")

# %% [markdown]
## 8. CE-Plane, CEAC & EVPI

print("\n--- Running Module 8: CE-Plane, CEAC & EVPI ---")
wtp = cp.linspace(0, 150000, 40)
dcost = psa_df["dCost"].to_cupy()
dqaly = psa_df["dQALY"].to_cupy()

# Filter out non-positive QALYs for stable NB calculation
valid_psa = dqaly > 0
nb = dqaly[valid_psa, None] * wtp[None, :] - dcost[valid_psa, None]

# CE-Plane scatter
plt.figure()
plt.scatter(dqaly.get(), dcost.get(), alpha=0.6)
plt.xlabel("Incremental QALYs (Delta QALY)")
plt.ylabel("Incremental Cost (Delta Cost)")
plt.title("Cost-Effectiveness Plane")
plt.grid(True)
plt.show()

# CEAC
prob = (nb > 0).mean(axis=0)
plt.figure()
plt.plot(wtp.get(), prob.get())
plt.title("Cost-Effectiveness Acceptability Curve (CEAC)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Probability Cost-Effective")
plt.ylim(0, 1)
plt.grid(True)
plt.show()

# EVPI - Corrected formula
evpi = cp.mean(cp.maximum(0, nb), axis=0) - cp.maximum(0, cp.mean(nb, axis=0))
plt.figure()
plt.plot(wtp.get(), evpi.get())
plt.title("Expected Value of Perfect Information (EVPI)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("EVPI")
plt.grid(True)
plt.show()

# %% [markdown]
## 9. One-Way Sensitivity (Tornado)

print("\n--- Running Module 9: One-Way Sensitivity (Tornado) ---")
tornado = []
base_params = {"familial_prop": 0.165, "yield_wgs": defaul_yield}
# Calling run_sim without seed_idx
base_dc, base_dq = run_sim(base_params, pop_size=10000)
base_icer = base_dc / base_dq if base_dq != 0 else 0

for param in ["yield_wgs", "familial_prop"]:
    for change in [-0.2, 0.2]:
        p = base_params.copy()
        p[param] = p[param] * (1 + change)
        # Calling run_sim without seed_idx
        dc, dq = run_sim(p, pop_size=10000)
        icer = (dc / dq) if dq != 0 else 0
        tornado.append(
            {"param": param, "change": f"{int(change * 100)}%", "ICER": icer}
        )

tor_df = cudf.DataFrame(tornado)
labels = [
    f"{p} {c}"
    for p, c in zip(
        tor_df["param"].to_arrow().to_pylist(), tor_df["change"].to_arrow().to_pylist()
    )
]
icers = tor_df["ICER"].to_arrow().to_pylist()
bar_values = [(x - base_icer) for x in icers]
colors = ["red" if x < 0 else "green" for x in bar_values]

plt.figure()
plt.barh(labels, bar_values, color=colors)
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("Change in ICER from base (/QALY)")
plt.title("Tornado Sensitivity Analysis")
plt.tight_layout()
plt.show()

# %% [markdown]
## 10. Sub-group Analyses

print("\n--- Running Module 10: Sub-group Analyses ---")
sg = []

# 1. Severity
orig_sev_dist = SEVERITY_DIST.copy()
for sev, label in [(0, "Mild"), (1, "Moderate")]:
    SEVERITY_DIST = cp.array([1.0 if i == sev else 0.0 for i in range(2)])
    # Calling run_sim without seed_idx
    dc, dq = run_sim(
        {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=15000
    )
    sg.append({"group": f"Severity: {label}", "ICER": (dc / dq) if dq != 0 else 0})
SEVERITY_DIST = orig_sev_dist  # Restore

# 2. Age at test
for age in [0, 18, 30]:
    params = {"familial_prop": 0.165, "yield_wgs": defaul_yield, "min_test_age": age}
    # Calling run_sim without seed_idx
    dc, dq = run_sim(params, pop_size=15000)
    sg.append({"group": f"Min Test Age: {age}", "ICER": (dc / dq) if dq != 0 else 0})

# 3. Time-horizon
# Note: run_scenario still needs to be run sequentially as it collects detailed yearly data
(dc_full, dq_full), int_full = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=15000
)
dfp = int_full[int_full["arm"] == "prospective"]
dfr = int_full[int_full["arm"] == "retrospective"]
for yrs_n in [10, 20, len(YEARS)]:
    yr = START_YEAR + yrs_n - 1
    c1 = float(dfp.loc[dfp["year"] == yr, "costs"].values[0])
    c0 = float(dfr.loc[dfr["year"] == yr, "costs"].values[0])
    q1 = float(dfp.loc[dfp["year"] == yr, "qalys"].values[0])
    q0 = float(dfr.loc[dfr["year"] == yr, "qalys"].values[0])
    dc, dq = (c1 - c0), (q1 - q0)
    sg.append({"group": f"Horizon: {yrs_n} yrs", "ICER": (dc / dq) if dq != 0 else 0})

# Plot
sg_df = cudf.DataFrame(sg)
plt.figure()
plt.scatter(
    sg_df["ICER"].to_arrow().to_pylist(),
    sg_df["group"].to_arrow().to_pylist(),
    marker="o",
)
plt.axvline(
    base_icer, color="grey", linestyle="--", label=f"Base Case ICER ({base_icer:,.0f})"
)
plt.xlabel("ICER (/QALY)")
plt.title("Sub-group Analyses")
plt.legend()
plt.tight_layout()
plt.grid(axis="x")
plt.show()

# %% [markdown]
## 11. Cost-Mix Stacked Area (Prospective)

print("\n--- Running Module 11: Cost-Mix Stacked Area ---")
(_, _), mix = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
pm = mix[mix["arm"] == "prospective"]
cost_cols = [
    "cost_health",
    "cost_edu",
    "cost_respite",
    "cost_mobility",
    "cost_welfare",
    "cost_emp",
    "cost_morbidity",
]
years_host = pm["year"].to_arrow().to_pylist()
# Un-cumulate the data for stacking annual costs
data = [pm[col].to_arrow().to_pylist() for col in cost_cols]

plt.figure(figsize=(10, 6))
plt.stackplot(
    years_host, *data, labels=[c.replace("cost_", "").capitalize() for c in cost_cols]
)
plt.legend(loc="upper left")
plt.title("Cost Mix Over Time (Prospective Arm - Discounted Annual Costs)")
plt.xlabel("Year")
plt.ylabel("Annual Cost")
plt.show()

# %% [markdown]
## 12. 5-Year Budget Impact

print("\n--- Running Module 12: 5-Year Budget Impact ---")
budg = []
uptakes = [0.2, 0.5, 0.8]
yrs = YEARS[:5]
# Run simulation once to get the cost data
(_, _), df = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
dfp = df[df["arm"] == "prospective"]
dfr = df[df["arm"] == "retrospective"]

for u in uptakes:
    for yr in yrs:
        # Use 'annual_cost' which is not cumulative
        # Correct way to extract a single scalar from a cuDF Series
        c1 = float(dfp.loc[dfp["year"] == yr, "annual_cost"].to_cupy().item())
        c0 = float(dfr.loc[dfr["year"] == yr, "annual_cost"].to_cupy().item())

        # Budget impact is the incremental cost for the new intervention, scaled by uptake
        incremental_cost = c1 - c0
        budg.append({"uptake": u, "year": yr, "budget_impact": incremental_cost * u})

budg_df = cudf.DataFrame(budg)
plt.figure()
for u in uptakes:
    sub = budg_df[budg_df["uptake"] == u]
    yrs_h = sub["year"].to_arrow().to_pylist()
    vals = sub["budget_impact"].to_arrow().to_pylist()
    plt.plot(yrs_h, vals, label=f"Uptake {int(u * 100)}%", marker="o")
plt.title("5-Year Budget Impact")
plt.xlabel("Year")
plt.ylabel("Incremental Budget Impact")
plt.legend()
plt.grid(True)
plt.show()

# %% [markdown]
## 13. Equity by SES Quintile

print("\n--- Running Module 13: Equity by SES Quintile ---")
(_, _), eq_int = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
bas = eq_int[eq_int["arm"] == "prospective"]
# Select the last year of the simulation
row_series = bas[bas["year"] == YEARS[-1]].iloc[0]

ses_cols = [f"ses_q{i}_pct" for i in range(5)]
# Extract scalar values from the series by converting to cupy and then using .item()
ses_vals = [float(row_series[col].to_cupy().item()) for col in ses_cols]
labels = [f"Quintile {i + 1}" for i in range(5)]

plt.figure()
plt.bar(labels, ses_vals)
plt.title(f"SES Distribution of Population in Final Year ({YEARS[-1]})")
plt.xlabel("SES Quintile")
plt.ylabel("Percentage of Population (%)")
plt.tight_layout()
plt.show()

# %% [markdown]
## 14. PSA Histograms

print("\n--- Running Module 14: PSA Histograms ---")
cost_list = psa_df["dCost"].to_arrow().to_pylist()
qaly_list = psa_df["dQALY"].to_arrow().to_pylist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.hist(cost_list, bins=30, color="skyblue", edgecolor="black")
ax1.set_title("PSA Distribution of Incremental Cost")
ax1.set_xlabel("Delta Cost")
ax1.set_ylabel("Frequency")

ax2.hist(qaly_list, bins=30, color="salmon", edgecolor="black")
ax2.set_title("PSA Distribution of Incremental QALYs")
ax2.set_xlabel("Delta QALY")
ax2.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# %% [markdown]
## 15. CEAF (Cost-Effectiveness Acceptability Frontier)

print("\n--- Running Module 15: CEAF ---")
# Reuse 'prob' (CEAC probabilities) from Module 8
prob_host = prob.get()  # Get the CuPy array to host memory (NumPy array)
ceaf = np.maximum.accumulate(prob_host)  # Use numpy for the accumulate operation

plt.figure()
plt.plot(wtp.get(), prob.get(), label="CEAC")
plt.plot(wtp.get(), ceaf, label="CEAF", linestyle="--")  # ceaf is already a NumPy array
plt.title("CEAC and CEAF")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Probability")
plt.legend()
plt.grid(True)
plt.show()

# %% [markdown]
## 16. Input Parameter Summary

print("\n--- Running Module 16: Saving Input Parameter Summary ---")
params_summary = {
    "DISCOUNT": DISCOUNT,
    "default_yield": defaul_yield,
    "U_DIAG": U_DIAG,
    "U_CARER": U_CARER,
    "RESPITE": RESPITE,
    "MOBILITY": MOBILITY,
    "FERT_AGE_SPEC": json.dumps(
        {f"{k[0]}_{k[1][0]}_{k[1][1]}": v for k, v in FERT_AGE_SPEC.items()}
    ),
    "MORB_PROB": json.dumps(MORB_PROB),
    "AGE_DIST": json.dumps(AGE_DIST.get().tolist()),
    "COST_REFERENCE_YEAR": COST_REFERENCE_YEAR,
    "INFLATION_RATE": INFLATION_RATE,
    "INFLATION_FACTOR": float(INFLATION_FACTOR),
}

param_df = cudf.DataFrame(
    {
        "parameter": list(params_summary.keys()),
        "value": [str(v) for v in params_summary.values()],
    }
)

param_df.to_csv("input_summary.csv", index=False)
print("Input parameters saved to input_summary.csv")

# %% [markdown]
## 17. Publication Comparison

print("\n--- Running Module 17: Publication Comparison ---")
# Published point estimates for comparison
published = {"base": {"dCost": 480000, "dQALY": 6.5, "ICER": 73846}}

pub_rows = []
for scen, vals in published.items():
    pub_rows.append(
        {
            "scenario": scen,
            "dCost_pub": vals["dCost"],
            "dQALY_pub": vals["dQALY"],
            "ICER_pub": vals["ICER"],
        }
    )
pub_df = cudf.DataFrame(pub_rows)

# Merge with your deterministic simulation output (det_df from Module 6)
df_cmp = det_df.merge(pub_df, on="scenario", how="left")
df_cmp = df_cmp.rename(
    columns={"dCost": "dCost_sim", "dQALY": "dQALY_sim", "ICER": "ICER_sim"}
)

print("Comparison of Simulation vs. Published Data:")
print(df_cmp.to_pandas().to_string())  # Use pandas for cleaner console printing
df_cmp.to_csv("publication_comparison.csv", index=False)
print("Saved comparison to publication_comparison.csv")

# %% [markdown]
## 18. Familial ID Prevalence Sensitivity Analysis

print("\n--- Running Module 18: Familial ID Prevalence Sensitivity Analysis ---")
familial_prevalence_range = cp.linspace(0.10, 0.23, 10).get()  # Range from 10% to 23%
familial_sens_results = []
for f_prop in familial_prevalence_range:
    # Calling run_sim without seed_idx
    dc, dq = run_sim(
        {"familial_prop": f_prop, "yield_wgs": defaul_yield}, pop_size=10000
    )
    icer = (dc / dq) if dq != 0 else 0
    familial_sens_results.append(
        {"familial_prop": f_prop, "dCost": dc, "dQALY": dq, "ICER": icer}
    )

familial_sens_df = cudf.DataFrame(familial_sens_results)
print("Familial ID Prevalence Sensitivity Results:")
print(familial_sens_df.to_pandas().to_string())

plt.figure(figsize=(8, 6))
plt.plot(
    familial_sens_df["familial_prop"].to_arrow().to_pylist(),
    familial_sens_df["ICER"].to_arrow().to_pylist(),
    marker="o",
    linestyle="-",
)
plt.xlabel("Familial ID Prevalence")
plt.ylabel("ICER (/QALY)")
plt.title("Sensitivity to Familial ID Prevalence")
plt.grid(True)
plt.show()

# %% [markdown]
## 19. EVPPI Analysis

print("\n--- Running Module 19: EVPPI Analysis ---")


def calculate_evppi_curve(psa_df, param_name, wtp_range):
    """
    Calculates the EVPPI curve for a given parameter over a range of WTP values
    using a simplified regression-based approach.
    """
    evppi_values = []

    # Convert dCost and dQALY to CuPy arrays once for efficiency
    dcost_cupy = psa_df["dCost"].to_cupy()
    dqaly_cupy = psa_df["dQALY"].to_cupy()
    param_values_cupy = psa_df[param_name].to_cupy()

    # Create design matrix for linear regression (NMB ~ intercept + param_value)
    X = cp.vstack([cp.ones(len(psa_df)), param_values_cupy]).T

    for wtp_val in wtp_range:
        # Calculate NMB for the current WTP
        nmb_cupy = dqaly_cupy * wtp_val - dcost_cupy

        # Perform linear regression to get predicted NMB
        # lstsq returns (coefficients, residuals, rank, singular_values)
        coeffs = cp.linalg.lstsq(X, nmb_cupy, rcond=None)[0]
        nmb_predicted = X @ coeffs

        # EVPPI approximation: Mean of max(0, NMB) - Mean of max(0, predicted NMB)
        evppi_val = cp.mean(cp.maximum(0, nmb_cupy)) - cp.mean(
            cp.maximum(0, nmb_predicted)
        )
        evppi_values.append(evppi_val)

    return cp.array(evppi_values)


# Parameters for which to calculate EVPPI
parameters_for_evppi = ["familial_prop", "yield_wgs"]
evppi_results = {}

print("Calculating EVPPI for key parameters...")
for param in parameters_for_evppi:
    print(f"  - Calculating for: {param}")
    evppi_curve = calculate_evppi_curve(psa_df, param, wtp)
    evppi_results[param] = evppi_curve.get()  # Get to host for plotting

# Plotting EVPPI results
plt.figure(figsize=(10, 6))
for param, evppi_curve_host in evppi_results.items():
    plt.plot(
        wtp.get(), evppi_curve_host, label=f"EVPPI ({param.replace('_', ' ').title()})"
    )

plt.title("Expected Value of Partial Perfect Information (EVPPI) Curves")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("EVPPI (£)")
plt.legend()
plt.grid(True)
plt.show()

print("EVPPI analysis complete.")

## v4a5

In [42]:
import warnings

import cupy as cp

warnings.filterwarnings("ignore")
cp.random.seed(42)

# %% [markdown]
## 2. Global Parameters & GPU Constants

START_YEAR, END_YEAR = 2020, 2100
YEARS = list(range(START_YEAR, END_YEAR + 1))
DISCOUNT = 0.035
POP_SIZE = 62000

# Costs are based on 2017 values from the paper, adjust to START_YEAR (2020)
COST_REFERENCE_YEAR = 2017
INFLATION_RATE = 0.03
INFLATION_FACTOR = (1 + INFLATION_RATE) ** (START_YEAR - COST_REFERENCE_YEAR)

# Demographics
SEVERITY_DIST = cp.array([0.70, 0.30])
AGE_BREAKS = cp.arange(0, 95, 5)
AGE_DIST = cp.array(
    [
        0.064,
        0.067,
        0.069,
        0.069,
        0.071,
        0.076,
        0.080,
        0.077,
        0.080,
        0.079,
        0.073,
        0.078,
        0.077,
        0.072,
        0.055,
        0.041,
        0.031,
        0.022,
        0.012,
    ]
)
AGE_DIST /= AGE_DIST.sum()

# Mortality
LIFE_EXP_CP = cp.array([74, 54])
ABS_MORT = cp.array(
    [
        0.0002,
        0.0001,
        0.0001,
        0.0002,
        0.0003,
        0.0005,
        0.0007,
        0.0011,
        0.0019,
        0.0035,
        0.0055,
        0.0085,
        0.0130,
        0.0200,
        0.0350,
        0.0600,
        0.0950,
        0.1500,
        0.2500,
    ]
)
MORT_HR = cp.array([2.0, 4.0])

defaul_yield = 0.45

# Fertility bands
FERT_AGE_SPEC = {
    (0, (15, 19)): 0.003,
    (0, (20, 24)): 0.008,
    (0, (25, 29)): 0.010,
    (0, (30, 34)): 0.007,
    (0, (35, 45)): 0.002,
    (1, (15, 19)): 0.001,
    (1, (20, 24)): 0.003,
    (1, (25, 29)): 0.004,
    (1, (30, 34)): 0.003,
    (1, (35, 45)): 0.001,
}
# Build GPU lookup table for fertility rates
fert_data = []
for (s, (lo, hi)), r in FERT_AGE_SPEC.items():
    for age in range(lo, hi + 1):
        fert_data.append({"severity": s, "age": age, "rate": r})
FERT_RATES_GPU = cudf.DataFrame(fert_data)

# Utilities
U_BASE_CP = cp.array([0.78, 0.65])
U_DECAY = 0.0005
U_DIAG = 0.02
U_CARER = 0.05

# Additional parameters for WGS diagnostic pathway impact
WGS_DIAG_COST_ADJUSTMENT = -1000  # Placeholder: e.g., representing annual cost savings from fewer diagnostic tests due to WGS
WGS_DIAG_QALY_ADJUSTMENT = 0.01  # Placeholder: e.g., representing additional annual QALY gain from earlier/precise intervention due to WGS

# Costs - Adjusted for inflation from COST_REFERENCE_YEAR to START_YEAR
COST_INP_CP = cp.array([5000, 15000]) * INFLATION_FACTOR
COST_OUTP_CP = cp.array([1000, 3000]) * INFLATION_FACTOR
COST_PHARM_CP = cp.array([200, 500]) * INFLATION_FACTOR
EDU0 = cp.array([16107, 27903, 46733]) * INFLATION_FACTOR
EDU1 = cp.array([18958, 30789, 49604]) * INFLATION_FACTOR
RESPITE = 13024 * INFLATION_FACTOR
MOBILITY = 2493 * INFLATION_FACTOR
WELFARE_CP = cp.array([3000, 8000]) * INFLATION_FACTOR
EMP_CP = cp.array([0, 10000, 0]) * INFLATION_FACTOR

# Morbidity
MORB_KEYS = ["epilepsy", "asd", "mental"]
MORB_PROB = {"epilepsy": 0.25, "asd": 0.30, "mental": 0.35}
MORB_COST_CP = cp.array([2000, 1500, 1000]) * INFLATION_FACTOR
MORB_QALY_CP = cp.array([0.04, 0.03, 0.05])

# %% [markdown]
## 3. Cohort Generator (GPU)


def sample_ages_gpu(n):
    idx = cp.random.choice(len(AGE_BREAKS), n, p=AGE_DIST)
    return (AGE_BREAKS[idx] + cp.random.randint(0, 5, n)).astype(int)


def gen_population_gpu(n, familial_prop, arm="prospective"):
    ages = sample_ages_gpu(n)
    sex = cp.random.choice(2, n)
    sev = cp.random.choice(2, n, p=SEVERITY_DIST)
    familial = cp.random.random(n) < familial_prop
    diag = (
        cp.zeros(n, bool)
        if arm == "prospective"
        else (cp.random.random(n) < defaul_yield)
    )
    living = cp.random.choice(4, n, p=cp.array([0.2, 0.1, 0.3, 0.4]))
    employment = cp.random.choice(3, n, p=cp.array([0.7, 0.2, 0.1]))
    uses_respite = cp.random.random(n) < 0.3
    needs_mobility = cp.random.random(n) < 0.2
    school = cp.where(ages < 5, -1, cp.where(ages < 12, 0, 1))
    support = cp.random.choice(3, n, p=cp.array([0.5, 0.3, 0.2]))
    ses = cp.random.choice(5, n)
    df = cudf.DataFrame(
        {
            "age": ages,
            "sex": sex,
            "severity": sev,
            "familial": familial,
            "diagnosed": diag,
            "living": living,
            "employment": employment,
            "uses_respite": uses_respite,
            "needs_mobility": needs_mobility,
            "school": school,
            "support": support,
            "ses": ses,
        }
    )
    # Add columns for outputs
    for col in [
        "annual_cost",
        "annual_qaly",
        "cost_health",
        "cost_edu",
        "cost_respite",
        "cost_mobility",
        "cost_welfare",
        "cost_emp",
        "cost_morbidity",
    ]:
        df[col] = 0.0
    return df

# %% [markdown]
## 4. Population System (GPU)


class PopulationSystemGPU:
    def __init__(self, pop_size, familial_prop, arm="prospective"):
        self.arm = arm
        self.pop = gen_population_gpu(pop_size, familial_prop, arm)

    def _mortality(self):
        df = self.pop
        ages = df["age"].to_cupy()
        sev = df["severity"].to_cupy()
        # Ensure age index is within bounds for ABS_MORT
        age_indices = cp.clip((ages // 5).astype(int), 0, len(ABS_MORT) - 1)
        surv = ABS_MORT[age_indices] * MORT_HR[sev]
        mask = (cp.random.random(len(df)) > surv) & (ages < LIFE_EXP_CP[sev])
        self.pop = df.iloc[cp.where(mask)[0]].reset_index(drop=True)

    def _fertility(self, familial_prop):
        df = self.pop
        moms = df[(df["sex"] == 0) & df["age"].between(15, 45)]
        if moms.empty:
            return
        mr = moms.merge(FERT_RATES_GPU, on=["severity", "age"], how="left").fillna(0)
        rates = mr["rate"].values
        births = cp.random.random(len(rates)) < rates
        idx = cp.nonzero(births)[0]
        if len(idx) > 0:
            babies = gen_population_gpu(len(idx), familial_prop, self.arm)
            babies["age"] = 0
            # Inherit 'familial' status from the mother
            flags = mr["familial"].iloc[idx.get()].reset_index(drop=True)
            babies["familial"] = flags
            self.pop = cudf.concat([df, babies], ignore_index=True)

    def _diagnosis(self, params):
        if self.arm != "prospective":
            return
        df = self.pop
        mask = (
            (~df["diagnosed"])
            & df["familial"]
            & (df["age"] >= params.get("min_test_age", 0))
        )
        cnt = int(mask.sum())
        if cnt > 0:
            # Use .loc with a boolean mask to assign values
            indices = df.index[mask]
            df.loc[indices, "diagnosed"] = cp.random.random(cnt) < params["yield_wgs"]
        self.pop = df

    def _cost_qaly(self, year, params):
        df = self.pop
        n = len(df)
        if n == 0:
            return

        disc = 1 / ((1 + DISCOUNT) ** (year - START_YEAR))
        sev = df["severity"].to_cupy()
        ages = df["age"].to_cupy()

        # Costs
        hc = (COST_INP_CP[sev] + COST_OUTP_CP[sev] + COST_PHARM_CP[sev]) * disc
        rc = df["uses_respite"].to_cupy().astype(int) * RESPITE * disc
        mc = (
            -df["needs_mobility"].to_cupy().astype(int) * MOBILITY * disc
        )  # Cost saving
        wc = WELFARE_CP[sev] * disc
        emp = EMP_CP[df["employment"].to_cupy()] * disc

        ec = cp.zeros(n, dtype=cp.float64)
        m0 = df["school"] == 0
        m1 = df["school"] == 1
        if not df.loc[m0].empty:
            ec[m0.to_cupy()] = EDU0[df.loc[m0, "support"].to_cupy()] * disc
        if not df.loc[m1].empty:
            ec[m1.to_cupy()] = EDU1[df.loc[m1, "support"].to_cupy()] * disc

        morb_cost = cp.zeros(n, dtype=cp.float64)
        base_rand = cp.random.random(n)
        for i, k in enumerate(MORB_KEYS):
            m = base_rand < MORB_PROB[k]
            morb_cost += m.astype(cp.float64) * MORB_COST_CP[i] * disc

        # Apply WGS-specific cost adjustment if in prospective arm and diagnosed
        if self.arm == "prospective":
            m = df["diagnosed"].to_cupy()
            morb_cost += m.astype(cp.float64) * WGS_DIAG_COST_ADJUSTMENT * disc

        # QALYs
        morb_qaly = cp.zeros(n, dtype=cp.float64)
        for i, k in enumerate(MORB_KEYS):
            m = base_rand < MORB_PROB[k]  # Use same random numbers for consistency
            morb_qaly += m.astype(cp.float64) * MORB_QALY_CP[i] * disc

        uq = U_BASE_CP[sev] * cp.exp(-U_DECAY * cp.clip(ages - 20, 0, None))
        total_qaly = (
            uq + df["diagnosed"].to_cupy().astype(int) * U_DIAG + U_CARER
        ) * disc - morb_qaly

        # Apply WGS-specific QALY adjustment if in prospective arm and diagnosed
        if self.arm == "prospective":
            m = df["diagnosed"].to_cupy()
            total_qaly += m.astype(cp.float64) * WGS_DIAG_QALY_ADJUSTMENT * disc

        # Assign back to DataFrame
        df["cost_health"] = hc
        df["cost_edu"] = ec
        df["cost_respite"] = rc
        df["cost_mobility"] = mc
        df["cost_welfare"] = wc
        df["cost_emp"] = emp
        df["cost_morbidity"] = morb_cost
        df["annual_cost"] = hc + ec + rc + mc + wc + emp + morb_cost
        df["annual_qaly"] = total_qaly
        self.pop = df

    def step(self, year, params):
        self._mortality()
        self._fertility(params["familial_prop"])
        self._diagnosis(params)
        self.pop["age"] += 1
        self._cost_qaly(year, params)

# %% [markdown]
## 5. Run Helpers


def run_sim(params, pop_size=POP_SIZE, seed_idx=None):
    """Fast runner for aggregate results (dCost, dQALY). Used for PSA, Tornado.
    Accepts an optional seed_idx for probabilistic runs."""
    if seed_idx is not None:
        cp.random.seed(42 + seed_idx)
    else:
        # Default seed for deterministic/non-PSA runs for reproducibility
        cp.random.seed(42)

    sim_r = PopulationSystemGPU(pop_size, params["familial_prop"], "retrospective")
    sim_p = PopulationSystemGPU(pop_size, params["familial_prop"], "prospective")

    cum_r_c = cum_r_q = cum_p_c = cum_p_q = 0.0

    for yr in YEARS:
        sim_r.step(yr, params)
        if not sim_r.pop.empty:
            cum_r_c += float(sim_r.pop["annual_cost"].sum())
            cum_r_q += float(sim_r.pop["annual_qaly"].sum())

        sim_p.step(yr, params)
        if not sim_p.pop.empty:
            cum_p_c += float(sim_p.pop["annual_cost"].sum())
            cum_p_q += float(sim_p.pop["annual_qaly"].sum())

    return (cum_p_c - cum_r_c, cum_p_q - cum_r_q)


def run_scenario(params, pop_size=POP_SIZE):
    """Detailed runner that returns aggregate results AND a per-year DataFrame."""
    outputs = []
    final_totals = {}

    for arm in ["retrospective", "prospective"]:
        sim = PopulationSystemGPU(pop_size, params["familial_prop"], arm)
        cum_c = 0.0
        cum_q = 0.0
        for yr in YEARS:
            sim.step(yr, params)
            df = sim.pop

            if df.empty:
                # Append a zero-row to maintain structure
                row_data = {
                    "arm": arm,
                    "year": yr,
                    "annual_cost": 0.0,
                    "cost_health": 0.0,
                    "cost_edu": 0.0,
                    "cost_respite": 0.0,
                    "cost_mobility": 0.0,
                    "cost_welfare": 0.0,
                    "cost_emp": 0.0,
                    "cost_morbidity": 0.0,
                    "costs": cum_c,
                    "qalys": cum_q,
                    **{f"ses_q{i}_pct": 0.0 for i in range(5)},
                }
                outputs.append(row_data)
                continue

            # Extract already-discounted annual components for detailed output
            hc = float(df["cost_health"].sum())
            edu = float(df["cost_edu"].sum())
            rc = float(df["cost_respite"].sum())
            mob = float(df["cost_mobility"].sum())
            wc = float(df["cost_welfare"].sum())
            emp = float(df["cost_emp"].sum())
            mc = float(df["cost_morbidity"].sum())

            annual_c_sum = float(df["annual_cost"].sum())
            annual_q_sum = float(df["annual_qaly"].sum())

            cum_c += annual_c_sum
            cum_q += annual_q_sum

            outputs.append(
                {
                    "arm": arm,
                    "year": yr,
                    "annual_cost": annual_c_sum,  # Crucial for budget impact
                    "cost_health": hc,
                    "cost_edu": edu,
                    "cost_respite": rc,
                    "cost_mobility": mob,
                    "cost_welfare": wc,
                    "cost_emp": emp,
                    "cost_morbidity": mc,
                    "costs": cum_c,
                    "qalys": cum_q,
                    **{
                        f"ses_q{i}_pct": float((df["ses"] == i).mean() * 100)
                        for i in range(5)
                    },
                }
            )

        final_totals[arm] = {"cost": cum_c, "qaly": cum_q}

    d_cost = final_totals["prospective"]["cost"] - final_totals["retrospective"]["cost"]
    d_qaly = final_totals["prospective"]["qaly"] - final_totals["retrospective"]["qaly"]

    return (d_cost, d_qaly), cudf.DataFrame(outputs)

# %% [markdown]
## 6. Deterministic Scenarios

print("--- Running Module 6: Deterministic Scenarios ---")
scenarios = {"low": 0.10, "base": 0.165, "high": 0.23}
results = []
for name, val in scenarios.items():
    # Calling run_sim without seed_idx
    dc, dq = run_sim({"familial_prop": val, "yield_wgs": defaul_yield}, pop_size=20000)
    icer = (dc / dq) if dq != 0 else 0
    results.append({"scenario": name, "dCost": dc, "dQALY": dq, "ICER": icer})

det_df = cudf.DataFrame(results)
det_df.to_csv("deterministic_summary.csv", index=False)
print(det_df)

plt.figure()
plt.bar(
    det_df["scenario"].to_arrow().to_pylist(), det_df["ICER"].to_arrow().to_pylist()
)
plt.title("Deterministic ICERs (/QALY)")
plt.ylabel("ICER (/QALY)")
plt.show()

# %% [markdown]
## 7. Probabilistic Sensitivity Analysis (PSA)

print("\n--- Running Module 7: Probabilistic Sensitivity Analysis ---")
psa_n = 300
psa_out = []

# Initialize Dask-CUDA cluster for parallel processing
cluster = LocalCUDACluster()
client = Client(cluster)

print(f"Dask dashboard link: {client.dashboard_link}")
print(f"Running {psa_n} PSA simulations in parallel using Dask-CUDA...")

# Prepare arguments for parallel map
psa_args = []
for i in range(psa_n):
    samp = {"familial_prop": beta.rvs(30, 150), "yield_wgs": beta.rvs(45, 55)}
    # Store the sampled parameters along with the run arguments
    psa_args.append((samp, 8000, i))  # (params, pop_size, seed_idx)

# Map the run_sim function across the cluster
futures = client.map(lambda args: run_sim(args[0], args[1], args[2]), psa_args)
progress(futures)  # Show progress bar

# Gather results
results = client.gather(futures)

# Populate psa_out from gathered results, including sampled parameters
for i, (dc, dq) in enumerate(results):
    # Retrieve the original sampled parameters for this run from psa_args
    original_samp = psa_args[i][0]
    psa_out.append(
        {
            "run": i,
            "dCost": dc,
            "dQALY": dq,
            "familial_prop": original_samp["familial_prop"],  # Store sampled value
            "yield_wgs": original_samp["yield_wgs"],  # Store sampled value
        }
    )

psa_df = cudf.DataFrame(psa_out)
psa_df.to_csv("psa_samples.csv", index=False)
print("PSA results saved to psa_samples.csv")

# Close the Dask client and cluster
client.close()
cluster.close()
print("Dask cluster closed.")

# %% [markdown]
## 8. CE-Plane, CEAC, EVPI & ENMB

print("\n--- Running Module 8: CE-Plane, CEAC, EVPI & ENMB ---")
wtp = cp.linspace(0, 150000, 40)
dcost = psa_df["dCost"].to_cupy()
dqaly = psa_df["dQALY"].to_cupy()

# Filter out non-positive QALYs for stable NB calculation
valid_psa = dqaly > 0
nb = dqaly[valid_psa, None] * wtp[None, :] - dcost[valid_psa, None]

# CE-Plane scatter with Quadrant Analysis
plt.figure()
plt.scatter(dqaly.get(), dcost.get(), alpha=0.6)
plt.axvline(x=0, color="grey", linestyle="--", linewidth=0.8)  # X-axis for dQALY=0
plt.axhline(y=0, color="grey", linestyle="--", linewidth=0.8)  # Y-axis for dCost=0
# Add WTP thresholds
plt.plot(
    wtp.get(), wtp.get() * 20000, color="green", linestyle=":", label="WTP £20,000/QALY"
)
plt.plot(
    wtp.get(), wtp.get() * 30000, color="red", linestyle=":", label="WTP £30,000/QALY"
)
plt.xlabel("Incremental QALYs (Delta QALY)")
plt.ylabel("Incremental Cost (Delta Cost)")
plt.title("Cost-Effectiveness Plane with WTP Thresholds")
plt.grid(True)
plt.legend()
plt.show()

# CEAC
prob = (nb > 0).mean(axis=0)
plt.figure()
plt.plot(wtp.get(), prob.get())
plt.title("Cost-Effectiveness Acceptability Curve (CEAC)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Probability Cost-Effective")
plt.ylim(0, 1)
plt.grid(True)
plt.show()

# EVPI - Corrected formula
evpi = cp.mean(cp.maximum(0, nb), axis=0) - cp.maximum(0, cp.mean(nb, axis=0))
plt.figure()
plt.plot(wtp.get(), evpi.get())
plt.title("Expected Value of Perfect Information (EVPI)")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("EVPI")
plt.grid(True)
plt.show()

# ENMB (Expected Net Monetary Benefit) Curve
enmb = cp.mean(nb, axis=0)
plt.figure()
plt.plot(wtp.get(), enmb.get())
plt.axhline(y=0, color="grey", linestyle="--", linewidth=0.8)
plt.title("Expected Net Monetary Benefit (ENMB) Curve")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Expected Net Monetary Benefit (£)")
plt.grid(True)
plt.show()

# %% [markdown]
## 9. One-Way Sensitivity (Tornado)

print("\n--- Running Module 9: One-Way Sensitivity (Tornado) ---")
tornado = []
base_params = {"familial_prop": 0.165, "yield_wgs": defaul_yield}
# Calling run_sim without seed_idx
base_dc, base_dq = run_sim(base_params, pop_size=10000)
base_icer = base_dc / base_dq if base_dq != 0 else 0

for param in ["yield_wgs", "familial_prop"]:
    for change in [-0.2, 0.2]:
        p = base_params.copy()
        p[param] = p[param] * (1 + change)
        # Calling run_sim without seed_idx
        dc, dq = run_sim(p, pop_size=10000)
        icer = (dc / dq) if dq != 0 else 0
        tornado.append(
            {"param": param, "change": f"{int(change * 100)}%", "ICER": icer}
        )

tor_df = cudf.DataFrame(tornado)
labels = [
    f"{p} {c}"
    for p, c in zip(
        tor_df["param"].to_arrow().to_pylist(), tor_df["change"].to_arrow().to_pylist()
    )
]
icers = tor_df["ICER"].to_arrow().to_pylist()
bar_values = [(x - base_icer) for x in icers]
colors = ["red" if x < 0 else "green" for x in bar_values]

plt.figure()
plt.barh(labels, bar_values, color=colors)
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("Change in ICER from base (/QALY)")
plt.title("Tornado Sensitivity Analysis")
plt.tight_layout()
plt.show()

# %% [markdown]
## 10. Sub-group Analyses

print("\n--- Running Module 10: Sub-group Analyses ---")
sg = []

# 1. Severity
orig_sev_dist = SEVERITY_DIST.copy()
for sev, label in [(0, "Mild"), (1, "Moderate")]:
    SEVERITY_DIST = cp.array([1.0 if i == sev else 0.0 for i in range(2)])
    # Calling run_sim without seed_idx
    dc, dq = run_sim(
        {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=15000
    )
    sg.append({"group": f"Severity: {label}", "ICER": (dc / dq) if dq != 0 else 0})
SEVERITY_DIST = orig_sev_dist  # Restore

# 2. Age at test
for age in [0, 18, 30]:
    params = {"familial_prop": 0.165, "yield_wgs": defaul_yield, "min_test_age": age}
    # Calling run_sim without seed_idx
    dc, dq = run_sim(params, pop_size=15000)
    sg.append({"group": f"Min Test Age: {age}", "ICER": (dc / dq) if dq != 0 else 0})

# 3. Time-horizon
# Note: run_scenario still needs to be run sequentially as it collects detailed yearly data
(dc_full, dq_full), int_full = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=15000
)
dfp = int_full[int_full["arm"] == "prospective"]
dfr = int_full[int_full["arm"] == "retrospective"]
for yrs_n in [10, 20, len(YEARS)]:
    yr = START_YEAR + yrs_n - 1
    c1 = float(dfp.loc[dfp["year"] == yr, "costs"].values[0])
    c0 = float(dfr.loc[dfr["year"] == yr, "costs"].values[0])
    q1 = float(dfp.loc[dfp["year"] == yr, "qalys"].values[0])
    q0 = float(dfr.loc[dfr["year"] == yr, "qalys"].values[0])
    dc, dq = (c1 - c0), (q1 - q0)
    sg.append({"group": f"Horizon: {yrs_n} yrs", "ICER": (dc / dq) if dq != 0 else 0})

# Plot
sg_df = cudf.DataFrame(sg)
plt.figure()
plt.scatter(
    sg_df["ICER"].to_arrow().to_pylist(),
    sg_df["group"].to_arrow().to_pylist(),
    marker="o",
)
plt.axvline(
    base_icer, color="grey", linestyle="--", label=f"Base Case ICER ({base_icer:,.0f})"
)
plt.xlabel("ICER (/QALY)")
plt.title("Sub-group Analyses")
plt.legend()
plt.tight_layout()
plt.grid(axis="x")
plt.show()

# %% [markdown]
## 11. Cost-Mix Stacked Area (Prospective)

print("\n--- Running Module 11: Cost-Mix Stacked Area (Prospective) ---")
(_, _), mix_data_prospective = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
pm = mix_data_prospective[mix_data_prospective["arm"] == "prospective"]
cost_cols = [
    "cost_health",
    "cost_edu",
    "cost_respite",
    "cost_mobility",
    "cost_welfare",
    "cost_emp",
    "cost_morbidity",
]
years_host = pm["year"].to_arrow().to_pylist()
# Un-cumulate the data for stacking annual costs
data = [pm[col].to_arrow().to_pylist() for col in cost_cols]

plt.figure(figsize=(10, 6))
plt.stackplot(
    years_host, *data, labels=[c.replace("cost_", "").capitalize() for c in cost_cols]
)
plt.legend(loc="upper left")
plt.title("Cost Mix Over Time (Prospective Arm - Discounted Annual Costs)")
plt.xlabel("Year")
plt.ylabel("Annual Cost")
plt.show()

# %% [markdown]
## 12. 5-Year Budget Impact

print("\n--- Running Module 12: 5-Year Budget Impact ---")
budg = []
uptakes = [0.2, 0.5, 0.8]
yrs = YEARS[:5]
# Run simulation once to get the cost data
(_, _), df = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
dfp = df[df["arm"] == "prospective"]
dfr = df[df["arm"] == "retrospective"]

for u in uptakes:
    for yr in yrs:
        # Use 'annual_cost' which is not cumulative
        # Correct way to extract a single scalar from a cuDF Series
        c1 = float(dfp.loc[dfp["year"] == yr, "annual_cost"].to_cupy().item())
        c0 = float(dfr.loc[dfr["year"] == yr, "annual_cost"].to_cupy().item())

        # Budget impact is the incremental cost for the new intervention, scaled by uptake
        incremental_cost = c1 - c0
        budg.append({"uptake": u, "year": yr, "budget_impact": incremental_cost * u})

budg_df = cudf.DataFrame(budg)
plt.figure()
for u in uptakes:
    sub = budg_df[budg_df["uptake"] == u]
    yrs_h = sub["year"].to_arrow().to_pylist()
    vals = sub["budget_impact"].to_arrow().to_pylist()
    plt.plot(yrs_h, vals, label=f"Uptake {int(u * 100)}%", marker="o")
plt.title("5-Year Budget Impact")
plt.xlabel("Year")
plt.ylabel("Incremental Budget Impact")
plt.legend()
plt.grid(True)
plt.show()

# %% [markdown]
## 13. Equity by SES Quintile

print("\n--- Running Module 13: Equity by SES Quintile ---")
(_, _), eq_int = run_scenario(
    {"familial_prop": 0.165, "yield_wgs": defaul_yield}, pop_size=20000
)
bas = eq_int[eq_int["arm"] == "prospective"]
# Select the last year of the simulation
row_series = bas[bas["year"] == YEARS[-1]].iloc[0]

ses_cols = [f"ses_q{i}_pct" for i in range(5)]
# Extract scalar values from the series by converting to cupy and then using .item()
ses_vals = [float(row_series[col].to_cupy().item()) for col in ses_cols]
labels = [f"Quintile {i + 1}" for i in range(5)]

plt.figure()
plt.bar(labels, ses_vals)
plt.title(f"SES Distribution of Population in Final Year ({YEARS[-1]})")
plt.xlabel("SES Quintile")
plt.ylabel("Percentage of Population (%)")
plt.tight_layout()
plt.show()

# %% [markdown]
## 14. PSA Histograms

print("\n--- Running Module 14: PSA Histograms (dCost, dQALY) ---")
cost_list = psa_df["dCost"].to_arrow().to_pylist()
qaly_list = psa_df["dQALY"].to_arrow().to_pylist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.hist(cost_list, bins=30, color="skyblue", edgecolor="black")
ax1.set_title("PSA Distribution of Incremental Cost")
ax1.set_xlabel("Delta Cost")
ax1.set_ylabel("Frequency")

ax2.hist(qaly_list, bins=30, color="salmon", edgecolor="black")
ax2.set_title("PSA Distribution of Incremental QALYs")
ax2.set_xlabel("Delta QALY")
ax2.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# %% [markdown]
## 15. CEAF (Cost-Effectiveness Acceptability Frontier)

print("\n--- Running Module 15: CEAF ---")
# Reuse 'prob' (CEAC probabilities) from Module 8
prob_host = prob.get()  # Get the CuPy array to host memory (NumPy array)
ceaf = np.maximum.accumulate(prob_host)  # Use numpy for the accumulate operation

plt.figure()
plt.plot(wtp.get(), prob.get(), label="CEAC")
plt.plot(wtp.get(), ceaf, label="CEAF", linestyle="--")  # ceaf is already a NumPy array
plt.title("CEAC and CEAF")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("Probability")
plt.legend()
plt.grid(True)
plt.show()

# %% [markdown]
## 16. Input Parameter Summary

print("\n--- Running Module 16: Saving Input Parameter Summary ---")
params_summary = {
    "DISCOUNT": DISCOUNT,
    "default_yield": defaul_yield,
    "U_DIAG": U_DIAG,
    "U_CARER": U_CARER,
    "RESPITE": RESPITE,
    "MOBILITY": MOBILITY,
    "FERT_AGE_SPEC": json.dumps(
        {f"{k[0]}_{k[1][0]}_{k[1][1]}": v for k, v in FERT_AGE_SPEC.items()}
    ),
    "MORB_PROB": json.dumps(MORB_PROB),
    "AGE_DIST": json.dumps(AGE_DIST.get().tolist()),
    "COST_REFERENCE_YEAR": COST_REFERENCE_YEAR,
    "INFLATION_RATE": INFLATION_RATE,
    "INFLATION_FACTOR": float(INFLATION_FACTOR),
}

param_df = cudf.DataFrame(
    {
        "parameter": list(params_summary.keys()),
        "value": [str(v) for v in params_summary.values()],
    }
)

param_df.to_csv("input_summary.csv", index=False)
print("Input parameters saved to input_summary.csv")

# %% [markdown]
## 17. Publication Comparison

print("\n--- Running Module 17: Publication Comparison ---")
# Published point estimates for comparison
published = {"base": {"dCost": 480000, "dQALY": 6.5, "ICER": 73846}}

pub_rows = []
for scen, vals in published.items():
    pub_rows.append(
        {
            "scenario": scen,
            "dCost_pub": vals["dCost"],
            "dQALY_pub": vals["dQALY"],
            "ICER_pub": vals["ICER"],
        }
    )
pub_df = cudf.DataFrame(pub_rows)

# Merge with your deterministic simulation output (det_df from Module 6)
df_cmp = det_df.merge(pub_df, on="scenario", how="left")
df_cmp = df_cmp.rename(
    columns={"dCost": "dCost_sim", "dQALY": "dQALY_sim", "ICER": "ICER_sim"}
)

print("Comparison of Simulation vs. Published Data:")
print(df_cmp.to_pandas().to_string())  # Use pandas for cleaner console printing
df_cmp.to_csv("publication_comparison.csv", index=False)
print("Saved comparison to publication_comparison.csv")

# %% [markdown]
## 18. Familial ID Prevalence Sensitivity Analysis

print("\n--- Running Module 18: Familial ID Prevalence Sensitivity Analysis ---")
familial_prevalence_range = cp.linspace(0.10, 0.23, 10).get()  # Range from 10% to 23%
familial_sens_results = []
for f_prop in familial_prevalence_range:
    # Calling run_sim without seed_idx
    dc, dq = run_sim(
        {"familial_prop": f_prop, "yield_wgs": defaul_yield}, pop_size=10000
    )
    icer = (dc / dq) if dq != 0 else 0
    familial_sens_results.append(
        {"familial_prop": f_prop, "dCost": dc, "dQALY": dq, "ICER": icer}
    )

familial_sens_df = cudf.DataFrame(familial_sens_results)
print("Familial ID Prevalence Sensitivity Results:")
print(familial_sens_df.to_pandas().to_string())

plt.figure(figsize=(8, 6))
plt.plot(
    familial_sens_df["familial_prop"].to_arrow().to_pylist(),
    familial_sens_df["ICER"].to_arrow().to_pylist(),
    marker="o",
    linestyle="-",
)
plt.xlabel("Familial ID Prevalence")
plt.ylabel("ICER (/QALY)")
plt.title("Sensitivity to Familial ID Prevalence")
plt.grid(True)
plt.show()

# %% [markdown]
## 19. EVPPI Analysis

print("\n--- Running Module 19: EVPPI Analysis ---")


def calculate_evppi_curve(psa_df, param_name, wtp_range):
    """
    Calculates the EVPPI curve for a given parameter over a range of WTP values
    using a simplified regression-based approach.
    """
    evppi_values = []

    # Convert dCost and dQALY to CuPy arrays once for efficiency
    dcost_cupy = psa_df["dCost"].to_cupy()
    dqaly_cupy = psa_df["dQALY"].to_cupy()
    param_values_cupy = psa_df[param_name].to_cupy()

    # Create design matrix for linear regression (NMB ~ intercept + param_value)
    X = cp.vstack([cp.ones(len(psa_df)), param_values_cupy]).T

    for wtp_val in wtp_range:
        # Calculate NMB for the current WTP
        nmb_cupy = dqaly_cupy * wtp_val - dcost_cupy

        # Perform linear regression to get predicted NMB
        # lstsq returns (coefficients, residuals, rank, singular_values)
        coeffs = cp.linalg.lstsq(X, nmb_cupy, rcond=None)[0]
        nmb_predicted = X @ coeffs

        # EVPPI approximation: Mean of max(0, NMB) - Mean of max(0, predicted NMB)
        evppi_val = cp.mean(cp.maximum(0, nmb_cupy)) - cp.mean(
            cp.maximum(0, nmb_predicted)
        )
        evppi_values.append(evppi_val)

    return cp.array(evppi_values)


# Parameters for which to calculate EVPPI
parameters_for_evppi = ["familial_prop", "yield_wgs"]
evppi_results = {}

print("Calculating EVPPI for key parameters...")
for param in parameters_for_evppi:
    print(f"  - Calculating for: {param}")
    evppi_curve = calculate_evppi_curve(psa_df, param, wtp)
    evppi_results[param] = evppi_curve.get()  # Get to host for plotting

# Plotting EVPPI results
plt.figure(figsize=(10, 6))
for param, evppi_curve_host in evppi_results.items():
    plt.plot(
        wtp.get(), evppi_curve_host, label=f"EVPPI ({param.replace('_', ' ').title()})"
    )

plt.title("Expected Value of Partial Perfect Information (EVPPI) Curves")
plt.xlabel("Willingness-to-Pay per QALY")
plt.ylabel("EVPPI (£)")
plt.legend()
plt.grid(True)
plt.show()

print("EVPPI analysis complete.")

# %% [markdown]
## 20. PSA ICER Distribution

print("\n--- Running Module 20: PSA ICER Distribution ---")

# Calculate ICER for each PSA sample
# Handle cases where dQALY is zero to avoid division by zero
psa_icers = []
for i in range(len(psa_df)):
    dc = float(psa_df.loc[i, "dCost"])
    dq = float(psa_df.loc[i, "dQALY"])
    if dq != 0:
        psa_icers.append(dc / dq)
    else:
        psa_icers.append(
            np.inf if dc > 0 else (-np.inf if dc < 0 else 0)
        )  # Assign infinity or zero appropriately

# Convert to NumPy array for plotting
psa_icers_np = np.array(psa_icers)

# Filter out infinities for histogram plotting, or handle them specifically if desired
finite_icers = psa_icers_np[np.isfinite(psa_icers_np)]

plt.figure(figsize=(8, 6))
sns.histplot(finite_icers, bins=50, kde=True, color="purple", edgecolor="black")
plt.title("PSA Distribution of Incremental Cost-Effectiveness Ratios (ICERs)")
plt.xlabel("ICER (£/QALY)")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

print("PSA ICER Distribution analysis complete.")

# %% [markdown]
## 21. Cost-Mix Stacked Area (Retrospective)

print("\n--- Running Module 21: Cost-Mix Stacked Area (Retrospective) ---")
# Reuse 'mix_data_prospective' which contains data for both arms
rm = mix_data_prospective[mix_data_prospective["arm"] == "retrospective"]
cost_cols = [
    "cost_health",
    "cost_edu",
    "cost_respite",
    "cost_mobility",
    "cost_welfare",
    "cost_emp",
    "cost_morbidity",
]
years_host_r = rm["year"].to_arrow().to_pylist()
data_r = [rm[col].to_arrow().to_pylist() for col in cost_cols]

plt.figure(figsize=(10, 6))
plt.stackplot(
    years_host_r,
    *data_r,
    labels=[c.replace("cost_", "").capitalize() for c in cost_cols],
)
plt.legend(loc="upper left")
plt.title("Cost Mix Over Time (Retrospective Arm - Discounted Annual Costs)")
plt.xlabel("Year")
plt.ylabel("Annual Cost")
plt.show()

print("Retrospective Cost Mix analysis complete.")